In [ ]:
import pandas as pd                      # Veri manipülasyonu için pandas
import numpy as np                       # Sayısal işlemler için numpy
import matplotlib.pyplot as plt          # Görselleştirme (temel)
import seaborn as sns                    # Görselleştirme (gelişmiş)
from sklearn.impute import SimpleImputer # Basit imputasyon için
from sklearn.preprocessing import StandardScaler # Ölçeklendirme
from sklearn.linear_model import LinearRegression # Basit regresyon modeli
from sklearn.model_selection import train_test_split, cross_val_score # Model doğrulama
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error # Model metrikleri


In [ ]:
import pandas as pd

# Load the newly uploaded CSV file
csv_path = "GII.csv"
df = pd.read_csv(csv_path, delimiter=';', encoding='latin1')    # latin1 (ISO-8859-1):
# Batı Avrupa dilleri için geliştirilmiş bir karakter setidir.
# Fransızca, Almanca, İspanyolca gibi dillerin özel karakterlerini destekler.

df.columns = df.columns.str.strip()  # Clean column names str.strip() → Her bir sütunda bulunan gereksiz boşlukları siler.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# =====================================================
# 1️⃣ VERİYİ YÜKLEME (GII.csv)
# =====================================================

df = pd.read_csv(
    "GII.csv",
    sep=';',
    encoding='latin1'
)

# Sütun isimlerini temizle
df.columns = df.columns.str.strip()

# Bozuk / anlamsız ülke isimlerini çıkar
df = df[~df["Country"].str.lower().isin(["seliforp", "xedni", "xednı", "index"])]

# =====================================================
# 2️⃣ SADECE SAYISAL DEĞİŞKENLER
# =====================================================

numeric_df = df.select_dtypes(include=["int64", "float64"])

# =====================================================
# 3️⃣ KORELASYON MATRİSİ – ALT ÜÇGEN
# =====================================================

corr_matrix = numeric_df.corr()

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

plt.figure(figsize=(11, 9))

sns.heatmap(
    corr_matrix,
    mask=mask,
    annot=True,
    cmap="plasma",     # Canlı ve kontrastlı
    fmt=".2f",
    linewidths=0.5,
    vmin=-1,
    vmax=1
)

plt.title(
    "Correlation Matrix (Lower Triangle) — GII Pillars",
    fontsize=13,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

# =====================================================
# 4️⃣ KORELASYON TABLOSU
# =====================================================

print("\n=== GII Pillars Correlation Matrix ===\n")
print(corr_matrix.round(3))


In [ ]:
# =============================
# PCA ANALYSIS (PC1–PC5 reported, PC1–PC7 fitted)
# GII.csv
# =============================

import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import seaborn as sns

# --------------------------------------------------------
# 1) Load Data
# --------------------------------------------------------

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")

df.columns = df.columns.str.strip()
df["Country"] = df["Country"].str.strip()

df = df[~df["Country"].str.lower().isin(["seliforp", "xedni", "index"])]

# --------------------------------------------------------
# 2) Numeric Variables
# --------------------------------------------------------

numeric_df = (
    df.drop(columns=["Country"])
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

countries = df.loc[numeric_df.index, "Country"]

# --------------------------------------------------------
# 3) Standardization
# --------------------------------------------------------

scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_df)

# --------------------------------------------------------
# 4) PCA → FIT WITH 7 COMPONENTS
# --------------------------------------------------------

pca = PCA(n_components=7)
pca_scores = pca.fit_transform(scaled_data)

# --------------------------------------------------------
# 5) Explained Variance (PC1–PC5)
# --------------------------------------------------------

variance_df = pd.DataFrame({
    "Component": [f"PC{i}" for i in range(1, 6)],
    "Explained Variance Ratio": pca.explained_variance_ratio_[:5],
    "Cumulative Variance (%)": np.cumsum(pca.explained_variance_ratio_[:5]) * 100
})

print("\n=== EXPLAINED VARIANCE (PC1–PC5) ===")
print(variance_df.round(4))

# --------------------------------------------------------
# 6) Loadings Matrix (PC1–PC5)
# --------------------------------------------------------

loadings = pd.DataFrame(
    pca.components_[:5].T,
    index=numeric_df.columns,
    columns=[f"PC{i}" for i in range(1, 6)]
)

print("\n=== PCA LOADINGS (PC1–PC5) ===")
print(loadings.round(3))

# --------------------------------------------------------
# 7) Important Loadings (|loading| ≥ 0.5)
# --------------------------------------------------------

important_loadings = loadings[loadings.abs() >= 0.5].dropna(how="all")

print("\n=== IMPORTANT LOADINGS (|loading| ≥ 0.5) ===")
print(important_loadings.round(3))

# --------------------------------------------------------
# 8) Loadings Heatmap (PC1–PC5)
# --------------------------------------------------------

plt.figure(figsize=(9, 7))

sns.heatmap(
    loadings,
    annot=True,
    cmap="viridis",
    center=0,
    fmt=".3f"
)

plt.title("PCA Loadings Heatmap (PC1–PC5) — GII Pillars", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

# --------------------------------------------------------
# 9) PCA Scatter (PC1 vs PC2)
# --------------------------------------------------------

pc_vars = pca.explained_variance_ratio_[:5] * 100

plt.figure(figsize=(12, 8))

plt.scatter(
    pca_scores[:, 0],
    pca_scores[:, 1],
    alpha=0.6
)

for i, country in enumerate(countries):
    plt.text(
        pca_scores[i, 0],
        pca_scores[i, 1],
        country,
        fontsize=8,
        alpha=0.75
    )

plt.axhline(0, linewidth=0.8)
plt.axvline(0, linewidth=0.8)

plt.xlabel(f"PC1 ({pc_vars[0]:.2f}%)")
plt.ylabel(f"PC2 ({pc_vars[1]:.2f}%)")

plt.title("Country-Level PCA Projection (PC1 vs PC2) — GII", fontsize=13, pad=20)

plt.figtext(
    0.5, 0.92,
    f"Explained Variance: "
    f"PC1={pc_vars[0]:.2f}%,  "
    f"PC2={pc_vars[1]:.2f}%,  "
    f"PC3={pc_vars[2]:.2f}%,  "
    f"PC4={pc_vars[3]:.2f}%,  "
    f"PC5={pc_vars[4]:.2f}%",
    ha="center",
    fontsize=11,
    fontweight="bold"
)

plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout(rect=[0, 0, 1, 0.90])
plt.show()


In [ ]:
# =========================================================
# FULL PCA ANALYSIS + UNIT CIRCLE + COUNTRY SCORES
# PC1–PC5 REPORTED, PC1–PC7 FITTED
# JOURNAL READY – SINGLE SCRIPT (GII)
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --------------------------------------------------------
# 1) Load Data
# --------------------------------------------------------

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()
df["Country"] = df["Country"].str.strip()

# Bozuk satırları temizle
df = df[~df["Country"].str.lower().isin(["seliforp", "xedni", "index"])]

# --------------------------------------------------------
# 2) Numeric Variables
# --------------------------------------------------------

numeric_df = (
    df.drop(columns=["Country"])
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

countries = df.loc[numeric_df.index, "Country"]

# --------------------------------------------------------
# 3) Standardization
# --------------------------------------------------------

scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_df)

# --------------------------------------------------------
# 4) PCA (FIT WITH 7 COMPONENTS)
# --------------------------------------------------------

pca = PCA(n_components=7)
pca_scores = pca.fit_transform(scaled_data)

# --------------------------------------------------------
# 5) Loadings (PC1–PC5)
# --------------------------------------------------------

loadings = pd.DataFrame(
    pca.components_[:5].T,
    index=numeric_df.columns,
    columns=[f"PC{i}" for i in range(1, 6)]
)

# --------------------------------------------------------
# 6) PCA UNIT CIRCLE + COUNTRY SCORES (PC1–PC5)
# --------------------------------------------------------

short_labels = {
    "Score": "GII",
    "Institutions": "INST",
    "Human capital and research": "HCR",
    "Infrastructure": "INF",
    "Market sophistication": "MARK",
    "Business sophistication": "BUS",
    "Knowledge and technology outputs": "KTO",
    "Creative outputs": "CRE"
}

eigenvalues = pca.explained_variance_

pc_pairs = [
    ("PC1", "PC2"),
    ("PC1", "PC3"),
    ("PC2", "PC3"),
    ("PC3", "PC4"),
    ("PC4", "PC5")
]

colors = plt.cm.tab10.colors

plt.figure(figsize=(20, 18))

for idx, (pc_x, pc_y) in enumerate(pc_pairs, 1):
    plt.subplot(3, 2, idx)

    # -----------------------------
    # Unit circle
    # -----------------------------
    theta = np.linspace(0, 2 * np.pi, 400)
    plt.plot(np.cos(theta), np.sin(theta), '--', color='gray', linewidth=1)
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)

    # -----------------------------
    # Country scores (normalized)
    # -----------------------------
    x_idx = int(pc_x[-1]) - 1
    y_idx = int(pc_y[-1]) - 1

    xs = pca_scores[:, x_idx]
    ys = pca_scores[:, y_idx]

    xs /= np.max(np.abs(xs))
    ys /= np.max(np.abs(ys))

    plt.scatter(xs, ys, alpha=0.4, color="black", s=12)

    # -----------------------------
    # Variable vectors + numeric labels
    # -----------------------------
    for i, var in enumerate(numeric_df.columns):
        lx = loadings.loc[var, pc_x]
        ly = loadings.loc[var, pc_y]

        x = lx * np.sqrt(eigenvalues[x_idx])
        y = ly * np.sqrt(eigenvalues[y_idx])

        plt.arrow(
            0, 0, x, y,
            color=colors[i % len(colors)],
            linewidth=2,
            head_width=0.035,
            head_length=0.035,
            length_includes_head=True
        )

        label = short_labels.get(var, var)

        plt.text(
            x * 1.05,
            y * 1.05,
            f"{label}\n({lx:.2f},{ly:.2f})",
            fontsize=6,
            fontweight="bold",
            ha="center",
            va="center"
        )

    # -----------------------------
    # Axis labels & titles
    # -----------------------------
    plt.xlabel(
        f"{pc_x} ({pca.explained_variance_ratio_[x_idx]*100:.2f}%)",
        fontsize=9,
        fontweight="bold"
    )

    plt.ylabel(
        f"{pc_y} ({pca.explained_variance_ratio_[y_idx]*100:.2f}%)",
        fontsize=9,
        fontweight="bold"
    )

    plt.title(
        f"PCA Correlation Circle: {pc_x} vs {pc_y} (GII)",
        fontsize=9,
        fontweight="bold"
    )

    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, linestyle="--", alpha=0.4)

plt.tight_layout()
plt.show()


In [ ]:
# =========================================================
# FULL PCA ANALYSIS + UNIT CIRCLE + COUNTRY SCORES
# PC1–PC5 REPORTED, PC1–PC7 FITTED
# GII.csv — JOURNAL READY (COMPACT FIGURE LAYOUT)
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --------------------------------------------------------
# 1) Load Data
# --------------------------------------------------------

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")

df.columns = df.columns.str.strip()
df["Country"] = df["Country"].str.strip()

df = df[~df["Country"].str.lower().isin(["seliforp", "xedni", "index"])]

# --------------------------------------------------------
# 2) Numeric Variables
# --------------------------------------------------------

numeric_df = (
    df.drop(columns=["Country"])
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

countries = df.loc[numeric_df.index, "Country"]

# --------------------------------------------------------
# 3) Standardization
# --------------------------------------------------------

scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_df)

# --------------------------------------------------------
# 4) PCA (FIT WITH 7 COMPONENTS)
# --------------------------------------------------------

pca = PCA(n_components=7)
pca_scores = pca.fit_transform(scaled_data)

# --------------------------------------------------------
# 5) LOADINGS MATRIX (PC1–PC5)
# --------------------------------------------------------

loadings = pd.DataFrame(
    pca.components_[:5].T,
    index=numeric_df.columns,
    columns=[f"PC{i}" for i in range(1, 6)]
)

eigenvalues = pca.explained_variance_

# --------------------------------------------------------
# 6) PRINT NUMERIC RESULTS
# --------------------------------------------------------

print("\n" + "="*70)
print("EXPLAINED VARIANCE (PC1–PC7)")
print("="*70)

for i in range(7):
    print(f"PC{i+1}: "
          f"Eigenvalue = {eigenvalues[i]:.4f} | "
          f"Variance Ratio = {pca.explained_variance_ratio_[i]*100:.2f}%")

print("\n" + "="*70)
print("PCA LOADINGS MATRIX (PC1–PC5)")
print("="*70)
print(loadings.round(4))

print("\n" + "="*70)
print("DETAILED NUMERIC LOADINGS PER VARIABLE")
print("="*70)

for var in loadings.index:
    print(f"\nVariable: {var}")
    for pc in loadings.columns:
        print(f"   {pc}: {loadings.loc[var, pc]:.4f}")

loadings.round(4).to_csv("PCA_Loadings_PC1_PC5.csv")

# --------------------------------------------------------
# 7) PCA UNIT CIRCLE (COMPACT LAYOUT)
# --------------------------------------------------------

short_labels = {
    "Score": "GII",
    "Institutions": "INST",
    "Human capital and research": "HCR",
    "Infrastructure": "INF",
    "Market sophistication": "MARK",
    "Business sophistication": "BUS",
    "Knowledge and technology outputs": "KTO",
    "Creative outputs": "CRE"
}

pc_pairs = [
    ("PC1", "PC2"),
    ("PC1", "PC3"),
    ("PC2", "PC3"),
    ("PC3", "PC4"),
    ("PC4", "PC5")
]

colors = plt.cm.tab10.colors

# 🔹 Daha kompakt figure boyutu
fig = plt.figure(figsize=(16, 14))

for idx, (pc_x, pc_y) in enumerate(pc_pairs, 1):
    ax = fig.add_subplot(3, 2, idx)

    theta = np.linspace(0, 2 * np.pi, 400)
    ax.plot(np.cos(theta), np.sin(theta), '--', color='gray', linewidth=1)
    ax.axhline(0, linewidth=0.8)
    ax.axvline(0, linewidth=0.8)

    x_idx = int(pc_x[-1]) - 1
    y_idx = int(pc_y[-1]) - 1

    xs = pca_scores[:, x_idx]
    ys = pca_scores[:, y_idx]

    xs /= np.max(np.abs(xs))
    ys /= np.max(np.abs(ys))

    ax.scatter(xs, ys, s=12, alpha=0.4, color="blue")

    for i, var in enumerate(numeric_df.columns):
        lx = loadings.loc[var, pc_x]
        ly = loadings.loc[var, pc_y]

        x = lx * np.sqrt(eigenvalues[x_idx])
        y = ly * np.sqrt(eigenvalues[y_idx])

        ax.arrow(
            0, 0, x, y,
            color=colors[i % len(colors)],
            linewidth=2,
            head_width=0.03,
            head_length=0.03,
            length_includes_head=True
        )

        label = short_labels.get(var, var)

        ax.text(
            x * 1.08,
            y * 1.08,
            f"{label}\n({lx:.2f}, {ly:.2f})",
            fontsize=6.5,
            fontweight="bold",
            ha="center",
            va="center"
        )

    ax.set_xlabel(
        f"{pc_x} ({pca.explained_variance_ratio_[x_idx]*100:.2f}%)",
        fontsize=9,
        fontweight="bold"
    )

    ax.set_ylabel(
        f"{pc_y} ({pca.explained_variance_ratio_[y_idx]*100:.2f}%)",
        fontsize=9,
        fontweight="bold"
    )

    ax.set_title(
        f"PCA Correlation Circle — {pc_x} vs {pc_y} (GII)",
        fontsize=10,
        fontweight="bold"
    )

    ax.set_aspect("equal", adjustable="box")
    ax.grid(True, linestyle="--", alpha=0.002)

# 🔹 BOŞLUKLARI DARALTAN KISIM
plt.tight_layout(pad=1.0)
plt.subplots_adjust(hspace=0.0015, wspace=0.0)

plt.show()

In [ ]:
# =========================================================
# PCA UNIT CIRCLE + COUNTRY SCORES (PC1–PC5)
# Ok ucunda etiket + sayısal yükler
# GII.csv — JOURNAL READY
# LAYOUT: 3 ROWS x 2 COLUMNS
# =========================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --------------------------------------------------------
# 1) Load Data
# --------------------------------------------------------

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()
df["Country"] = df["Country"].str.strip()

df = df[~df["Country"].str.lower().isin(["seliforp", "xedni", "index"])]

# --------------------------------------------------------
# 2) Numeric Variables
# --------------------------------------------------------

numeric_df = (
    df.drop(columns=["Country"])
    .apply(pd.to_numeric, errors="coerce")
    .dropna()
)

countries = df.loc[numeric_df.index, "Country"]

# --------------------------------------------------------
# 3) Standardization
# --------------------------------------------------------

scaler = StandardScaler()
scaled_data = scaler.fit_transform(numeric_df)

# --------------------------------------------------------
# 4) PCA (FIT WITH 7 COMPONENTS)
# --------------------------------------------------------

pca = PCA(n_components=7)
pca_scores = pca.fit_transform(scaled_data)

# --------------------------------------------------------
# 5) Loadings (PC1–PC5)
# --------------------------------------------------------

loadings = pd.DataFrame(
    pca.components_[:5].T,
    index=numeric_df.columns,
    columns=[f"PC{i}" for i in range(1, 6)]
)

# --------------------------------------------------------
# 6) PCA UNIT CIRCLE + COUNTRY SCORES (PC1–PC5)
# --------------------------------------------------------

short_labels = {
    "Score": "GII",
    "Institutions": "INST",
    "Human capital and research": "HCR",
    "Infrastructure": "INF",
    "Market sophistication": "MARK",
    "Business sophistication": "BUS",
    "Knowledge and technology outputs": "KTO",
    "Creative outputs": "CRE"
}

eigenvalues = pca.explained_variance_

pc_pairs = [
    ("PC1", "PC2"),
    ("PC1", "PC3"),
    ("PC2", "PC3"),
    ("PC3", "PC4"),
    ("PC4", "PC5")
]

colors = plt.cm.tab10.colors

plt.figure(figsize=(20, 18))

for idx, (pc_x, pc_y) in enumerate(pc_pairs, 1):
    plt.subplot(3, 2, idx)   # 🔹 3 SATIR × 2 SÜTUN

    # -----------------------------
    # Unit circle
    # -----------------------------
    theta = np.linspace(0, 2 * np.pi, 400)
    plt.plot(np.cos(theta), np.sin(theta), '--', color='gray', linewidth=1)
    plt.axhline(0, linewidth=1)
    plt.axvline(0, linewidth=1)

    # -----------------------------
    # Country scores (normalized)
    # -----------------------------
    x_idx = int(pc_x[-1]) - 1
    y_idx = int(pc_y[-1]) - 1

    xs = pca_scores[:, x_idx]
    ys = pca_scores[:, y_idx]

    xs /= np.max(np.abs(xs))
    ys /= np.max(np.abs(ys))

    plt.scatter(xs, ys, alpha=0.45, color="black", s=10)

    # -----------------------------
    # Variable vectors + labels at arrow tip
    # -----------------------------
    for i, var in enumerate(numeric_df.columns):
        lx = loadings.loc[var, pc_x]
        ly = loadings.loc[var, pc_y]

        x = lx * np.sqrt(eigenvalues[x_idx])
        y = ly * np.sqrt(eigenvalues[y_idx])

        plt.arrow(
            0, 0, x, y,
            color=colors[i % len(colors)],
            linewidth=1.5,
            head_width=0.03,
            head_length=0.03,
            length_includes_head=True
        )

        label = short_labels.get(var, var)

        angle = np.arctan2(y, x) + (i % 5 - 2) * 0.12
        r = np.sqrt(x**2 + y**2) * 1.01
        dx = r * np.cos(angle)
        dy = r * np.sin(angle)

        plt.text(
            dx,
            dy,
            f"{label}\n({lx:.2f},{ly:.2f})",
            fontsize=5,
            fontweight="bold",
            ha="center",
            va="center"
        )

    plt.xlabel(
        f"{pc_x} ({pca.explained_variance_ratio_[x_idx]*100:.2f}%)",
        fontsize=7,
        fontweight="bold"
    )

    plt.ylabel(
        f"{pc_y} ({pca.explained_variance_ratio_[y_idx]*100:.2f}%)",
        fontsize=7,
        fontweight="bold"
    )

    plt.title(
        f"PCA Correlation Circle: {pc_x} vs {pc_y} (GII)",
        fontsize=8,
        fontweight="bold"
    )

    plt.xticks(fontsize=6)
    plt.yticks(fontsize=6)
    plt.gca().set_aspect("equal", adjustable="box")
    plt.grid(True, linestyle="--", alpha=0.15)

# 6. panel boş kalsın (bilinçli)
plt.tight_layout()
plt.show()

# =========================================================
# PCA SAYISAL ÇIKTILAR
# =========================================================

print("=========== PCA SAYISAL ÖZET ===========")

print("\n-- Açıklanan Varyans (Eigenvalues) --")
for i, val in enumerate(pca.explained_variance_, 1):
    print(f"PC{i}: {val:.4f}")

print("\n-- Açıklanan Varyans Yüzdesi --")
for i, val in enumerate(pca.explained_variance_ratio_, 1):
    print(f"PC{i}: {val*100:.2f}%")

print("\n-- PCA Bileşen Yükleri (Loadings) --")
print(loadings.round(4))

pc_scores_df = pd.DataFrame(
    pca_scores,
    columns=[f"PC{i}" for i in range(1, 8)]
)
pc_scores_df["Country"] = countries.values
pc_scores_df.to_csv("pca_scores_GII.csv", index=False)


In [ ]:
# =====================================================
# 1️⃣ GEREKLİ KÜTÜPHANELER
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score


# =====================================================
# 2️⃣ VERİNİN OKUNMASI (⚠️ ; AYIRICI ÇOK KRİTİK)
# =====================================================

df = pd.read_csv(
    "GII.csv",
    sep=";",            # 🔥 EN KRİTİK SATIR
    encoding="latin1"
)

df.columns = df.columns.str.strip()
df["Country"] = df["Country"].str.strip()

# Bozuk satırları temizle
df = df[~df["Country"].str.lower().isin(["seliforp", "xedni", "index"])]

print("\nVeri seti boyutu:", df.shape)
print("Kolonlar:")
print(df.columns)


# =====================================================
# 3️⃣ SAYISAL DEĞİŞKENLERİN TEMİZLENMESİ (GII PILLARS)
# =====================================================

data = df.drop(columns=["Country"], errors="ignore")
data = data.apply(pd.to_numeric, errors="coerce")
data = data.dropna()

print("\nKullanılan sayısal değişkenler:")
print(list(data.columns))
print("Temiz veri boyutu:", data.shape)

if data.shape[0] == 0:
    raise ValueError("❌ Veri boş! GII.csv içeriğini kontrol et.")


# =====================================================
# 4️⃣ STANDARDIZATION
# =====================================================

scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)


# =====================================================
# 5️⃣ ELBOW & SILHOUETTE ANALİZİ
# =====================================================

inertia_values = []
silhouette_scores = []

K_values = range(2, 11)

for k in K_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )

    labels = kmeans.fit_predict(scaled_data)

    inertia_values.append(kmeans.inertia_)
    silhouette_scores.append(
        silhouette_score(scaled_data, labels)
    )


# =====================================================
# 6️⃣ SAYISAL SONUÇ TABLOSU
# =====================================================

print("\n=== ELBOW & SILHOUETTE SONUÇLARI (GII) ===")
print("{:<5} {:<15} {:<15}".format("K", "Inertia", "Silhouette"))

for k, i, s in zip(K_values, inertia_values, silhouette_scores):
    print(f"{k:<5} {i:<15.2f} {s:<15.4f}")


# =====================================================
# 7️⃣ BİRLEŞİK GRAFİK (ETİKETLİ – SSCI UYUMLU)
# =====================================================

fig, ax1 = plt.subplots(figsize=(10, 6))

# ---------------- Elbow (Inertia) ----------------
ax1.set_xlabel("K (Number of Clusters)", fontsize=11)
ax1.set_ylabel("Inertia", color="tab:blue", fontsize=11)
ax1.plot(
    K_values,
    inertia_values,
    "o-",
    color="tab:blue",
    label="Inertia"
)
ax1.tick_params(axis="y", labelcolor="tab:blue")

# Inertia etiketleri
for k, inertia in zip(K_values, inertia_values):
    ax1.text(
        k,
        inertia,
        f"{inertia:.2f}",
        color="tab:blue",
        fontsize=10,
        ha="center",
        va="bottom"
    )


# ---------------- Silhouette ----------------
ax2 = ax1.twinx()
ax2.set_ylabel("Silhouette Score", color="tab:red", fontsize=11)
ax2.plot(
    K_values,
    silhouette_scores,
    "s--",
    color="tab:red",
    label="Silhouette Score"
)
ax2.tick_params(axis="y", labelcolor="tab:red")

# Silhouette etiketleri
for k, sil in zip(K_values, silhouette_scores):
    ax2.text(
        k,
        sil,
        f"{sil:.4f}",
        color="tab:red",
        fontsize=10,
        ha="center",
        va="top"
    )


# ---------------- Örnek Optimal K ----------------
plt.axvline(
    x=6,
    color="green",
    linestyle="--",
    linewidth=2,
    label="Candidate K = 6"
)

fig.suptitle(
    "Elbow & Silhouette Analysis for GII Dataset",
    fontsize=13,
    fontweight="bold"
)

fig.tight_layout()

# Legend birleştirme
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
plt.legend(
    lines1 + lines2,
    labels1 + labels2,
    loc="upper right",
    fontsize=10
)

plt.show()


In [ ]:
# =====================================================
# 1️⃣ GEREKLİ KÜTÜPHANELER
# =====================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# =====================================================
# 2️⃣ VERİYİ OKU (GII CSV)
# =====================================================
csv_path = "GII.csv"
df = pd.read_csv(csv_path, sep=";", encoding="latin1")
df.columns = df.columns.str.strip()
df["Country"] = df["Country"].str.strip()

# Gereksiz/bozuk satırları temizle
df = df[~df["Country"].str.lower().isin(["seliforp", "xedni", "index"])]

print("Veri boyutu:", df.shape)
print(df.columns)

# =====================================================
# 3️⃣ SAYISAL DEĞİŞKENLERİ AL
# =====================================================
data = df.drop(columns=["Country"], errors="ignore")
data = data.apply(pd.to_numeric, errors="coerce").dropna()

# Aynı satırları df'de de tut
df = df.loc[data.index].reset_index(drop=True)
data = data.reset_index(drop=True)

print("Temiz veri boyutu:", data.shape)
print("Kullanılan sayısal değişkenler:", list(data.columns))

# =====================================================
# 4️⃣ STANDARDIZATION
# =====================================================
scaler = StandardScaler()
scaled_data = scaler.fit_transform(data)

# =====================================================
# 5️⃣ ELBOW & SILHOUETTE ANALİZİ (K=2..10)
# =====================================================
inertia_values = []
silhouette_scores = []

K_values = range(2, 11)

for k in K_values:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(scaled_data)
    inertia_values.append(kmeans.inertia_)
    silhouette_scores.append(silhouette_score(scaled_data, labels))

# =====================================================
# 6️⃣ SAYISAL SONUÇ TABLOSU
# =====================================================
print("\n=== ELBOW & SILHOUETTE SONUÇLARI (GII) ===")
print("{:<5} {:<15} {:<15}".format("K", "Inertia", "Silhouette"))
for k, i, s in zip(K_values, inertia_values, silhouette_scores):
    print(f"{k:<5} {i:<15.2f} {s:<15.4f}")

# =====================================================
# 7️⃣ K-MEANS (OPTIMAL K = 6) & LABELS
# =====================================================
optimal_k = 6
kmeans = KMeans(n_clusters=optimal_k, random_state=42, n_init=10)
labels = kmeans.fit_predict(scaled_data)
df["Cluster"] = labels

# =====================================================
# 8️⃣ PCA (2D) & GÖRSELLEŞTİRME
# =====================================================
pca = PCA(n_components=2)
pca_data = pca.fit_transform(scaled_data)

plt.figure(figsize=(16, 10))

scatter = sns.scatterplot(
    x=pca_data[:, 0],
    y=pca_data[:, 1],
    hue=df["Cluster"],
    palette="tab10",
    s=90,
    alpha=0.85
)

# Ülke isimlerini ekle
for i, country in enumerate(df["Country"]):
    plt.text(
        pca_data[i, 0] + 0.05,
        pca_data[i, 1] + 0.05,
        country,
        fontsize=8,
        alpha=0.75
    )

# Legend
cluster_counts = df["Cluster"].value_counts().sort_index()
legend_labels = [f"Cluster {i} (n={cluster_counts[i]} country)" for i in cluster_counts.index]
handles, _ = scatter.get_legend_handles_labels()
plt.legend(handles, legend_labels, title="Clusters")

plt.title(f"K-Means (K={optimal_k}) + PCA 2D Visualization for GII Dataset", fontsize=14)
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.grid(True)
plt.tight_layout()
plt.show()

# =====================================================
# 9️⃣ KÜMELERE GÖRE ÜLKELER
# =====================================================
print("\n=== KÜMELERE GÖRE ÜLKE DAĞILIMI ===")
for cluster_num in sorted(df["Cluster"].unique()):
    countries_in_cluster = df[df["Cluster"] == cluster_num]["Country"].tolist()
    print(f"\n--- Küme {cluster_num} ({len(countries_in_cluster)} ülke) ---")
    print(", ".join(countries_in_cluster))
    print("-" * 70)


Elbow analizinde K=5 dirsek noktasıdır, Silhouette analizinde K=5 çevresinde en dengeli ayrışma elde edilmektedir.  Bu değer, modelin hem yeterli ayrışmayı hem de istikrarlı kümelenmeyi sağladığı noktayı temsil etmektedir. Hem istatistiksel hem yorumlanabilirlik açısından K=5 optimal küme sayısıdır.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score

# -----------------------------------------------------
# 1️⃣ VERİNİN OKUNMASI
# -----------------------------------------------------
df = pd.read_csv(
    "GII.csv",
    sep=";",
    encoding="latin1"
)

df.columns = df.columns.str.strip()
df["Country"] = df["Country"].str.strip()

# Olası bozuk satırları temizle
df = df[~df["Country"].str.lower().isin(["index", "xedni", "seliforp"])]

print("Veri boyutu:", df.shape)
print(df.columns)

# -----------------------------------------------------
# 2️⃣ SAYISAL DEĞİŞKENLER (GII)
# -----------------------------------------------------
features = df.drop(columns=["Country"], errors="ignore").columns.tolist()

X = df[features].apply(pd.to_numeric, errors="coerce")
X = X.dropna()

# df ile senkronizasyon
df = df.loc[X.index].reset_index(drop=True)
X = X.reset_index(drop=True)

print("Temiz veri boyutu:", X.shape)

# -----------------------------------------------------
# 3️⃣ STANDARDIZATION
# -----------------------------------------------------
scaler = StandardScaler()
scaled_data = scaler.fit_transform(X)

# -----------------------------------------------------
# 4️⃣ K-MEANS MODELİ (K = 6)
# -----------------------------------------------------
k = 6

kmeans = KMeans(
    n_clusters=k,
    random_state=42,
    n_init=10
)

labels = kmeans.fit_predict(scaled_data)
centers = kmeans.cluster_centers_

# -----------------------------------------------------
# 5️⃣ KÜME ETİKETLERİNİ VERİYE EKLE
# -----------------------------------------------------
clustered_df = df.copy()
clustered_df["Cluster"] = labels

# -----------------------------------------------------
# 6️⃣ ÜLKE – KÜME EŞLEŞMELERİ
# -----------------------------------------------------
print("\n=== ÜLKE – KÜME EŞLEŞMELERİ ===")

for _, row in clustered_df.iterrows():
    print(f"{row['Country']:<30} → Küme {row['Cluster']}")

# -----------------------------------------------------
# 7️⃣ KÜME ORTALAMA DEĞERLERİ
# -----------------------------------------------------
cluster_means = (
    clustered_df
    .groupby("Cluster")[features]
    .mean()
    .round(2)
)

print("\n=== KÜMELERE GÖRE ORTALAMA GII DEĞERLERİ ===")
print(cluster_means)

# -----------------------------------------------------
# 8️⃣ HEATMAP — KÜME ORTALAMALARI
# -----------------------------------------------------
plt.figure(figsize=(12, 10))

sns.heatmap(
    cluster_means,
    annot=True,
    fmt=".2f",
    cmap="magma",
    linewidths=0.5,
    cbar_kws={"label": "Average Value"}
)

plt.title("Average GII Feature Values by Clusters (K=6)", fontsize=14)
plt.xlabel("GII Variables")
plt.ylabel("Clusters")
plt.tight_layout()
plt.show()

# -----------------------------------------------------
# 9️⃣ SİLHOUETTE SKORU
# -----------------------------------------------------
sil_score = silhouette_score(scaled_data, labels)

print(f"\nSilhouette Skoru (K=6): {sil_score:.4f}")


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd

# ---------------------------------------------------
# 1️⃣ Veri ve Özelliklerin Belirlenmesi (GII)
# ---------------------------------------------------
# clustered_df → K-Means sonrası oluşmuş (Country + GII + Cluster)

df_clean = clustered_df.copy()

# Country hariç tüm sayısal sütunları al
features = df_clean.select_dtypes(include=[np.number]).columns.tolist()

# Cluster’ı özellik listesinden çıkar
if "Cluster" in features:
    features.remove("Cluster")

# Kaç özellik var?
num_features = len(features)

print("Toplam GII değişken sayısı:", num_features)

# ---------------------------------------------------
# 2️⃣ Grafik Yerleşimi
# ---------------------------------------------------
cols = 2
rows = int(np.ceil(num_features / cols))

plt.figure(figsize=(18, rows * 8))
gs = gridspec.GridSpec(rows, cols)

# Küme sayısına göre renk paleti
palette = sns.color_palette("Set1", n_colors=df_clean["Cluster"].nunique())

# Her özellik için cluster ortalamalarını sakla
feature_means = {}

# ---------------------------------------------------
# 3️⃣ BOXPLOT + MEAN ANOTASYONLU GRAFİKLER
# ---------------------------------------------------
for idx, feature in enumerate(features):
    ax = plt.subplot(gs[idx])
    
    sns.boxplot(
        x="Cluster",
        y=feature,
        data=df_clean,
        palette="Set1",
        ax=ax
    )

    # Her küme için ortalama
    means = df_clean.groupby("Cluster")[feature].mean()
    feature_means[feature] = means

    # Ortalama değerleri grafiğe yaz
    for cluster_id, mean_value in means.items():
        ax.text(
            cluster_id,
            mean_value,
            f"{mean_value:.2f}",
            ha="center",
            va="bottom",
            fontsize=12,
            fontweight="bold",
            color="black"
        )

    ax.set_title(f"{feature} by Cluster (GII)", fontsize=12, fontweight="bold")
    ax.set_xlabel("Cluster")
    ax.set_ylabel("Value")

plt.tight_layout()
plt.show()

# ---------------------------------------------------
# 4️⃣ HER GII DEĞİŞKENİ İÇİN KÜME ORTALAMALARI
# ---------------------------------------------------
print("\n=== GII DEĞİŞKENLERİ – KÜME ORTALAMALARI ===\n")

for feature, means in feature_means.items():
    print(f"{feature} - Cluster Means:")
    print(means.round(2))
    print()


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics.pairwise import (
    euclidean_distances,
    cosine_similarity,
    manhattan_distances
)

# -------------------------------------------------------------
# 1️⃣ ROBUST CSV LOADER
# -------------------------------------------------------------
file_path = "GII.csv"

encodings = ["utf-8", "latin1", "cp1252"]
separators = [",", ";", "\t"]

loaded = False

for enc in encodings:
    for sep in separators:
        try:
            df = pd.read_csv(
                file_path,
                encoding=enc,
                sep=sep,
                engine="python",
                on_bad_lines="skip"
            )
            if df.shape[1] > 3:
                print(f"✅ Loaded with encoding={enc} | sep='{sep}'")
                loaded = True
                break
        except:
            continue
    if loaded:
        break

if not loaded:
    raise ValueError("⚠ File could not be parsed.")

print("\n📊 Data Shape:", df.shape)

# -------------------------------------------------------------
# 2️⃣ DATA CLEANING
# -------------------------------------------------------------
if "Country" in df.columns:
    df_numeric = df.drop(columns=["Country"])
else:
    df_numeric = df.copy()

df_numeric = df_numeric.select_dtypes(include=[np.number])

print("📊 Feature count:", df_numeric.shape[1])

# -------------------------------------------------------------
# 3️⃣ STANDARDIZATION (Z-SCORE)
# -------------------------------------------------------------
scaler = StandardScaler()
scaled_values = scaler.fit_transform(df_numeric)

scaled_df = pd.DataFrame(scaled_values, columns=df_numeric.columns)

# -------------------------------------------------------------
# 4️⃣ K-MEANS (k = 6)
# -------------------------------------------------------------
kmeans = KMeans(n_clusters=6, random_state=42, n_init=20)
scaled_df["Cluster"] = kmeans.fit_predict(scaled_df)

print("\n✅ K-Means clustering completed with k = 6.")

# -------------------------------------------------------------
# 5️⃣ CLUSTER MEANS
# -------------------------------------------------------------
features = df_numeric.columns.tolist()

cluster_means_z = (
    scaled_df
    .groupby("Cluster")[features]
    .mean()
    .round(4)
)

cluster_labels = [f"Cluster {i}" for i in cluster_means_z.index]

print("\n================ CLUSTER MEANS (Z) ================")
print(cluster_means_z)

# -------------------------------------------------------------
# 6️⃣ DISTANCE / SIMILARITY MATRICES
# -------------------------------------------------------------
euclid_matrix = pd.DataFrame(
    euclidean_distances(cluster_means_z),
    index=cluster_labels,
    columns=cluster_labels
)

manhattan_matrix = pd.DataFrame(
    manhattan_distances(cluster_means_z),
    index=cluster_labels,
    columns=cluster_labels
)

cosine_matrix = pd.DataFrame(
    cosine_similarity(cluster_means_z),
    index=cluster_labels,
    columns=cluster_labels
)

pearson_matrix = cluster_means_z.T.corr()
pearson_matrix.index = cluster_labels
pearson_matrix.columns = cluster_labels

# -------------------------------------------------------------
# 7️⃣ PRINT MATRICES
# -------------------------------------------------------------
print("\n================ EUCLIDEAN =================")
print(euclid_matrix.round(3))

print("\n================ MANHATTAN =================")
print(manhattan_matrix.round(3))

print("\n================ COSINE =================")
print(cosine_matrix.round(3))

print("\n================ PEARSON =================")
print(pearson_matrix.round(3))

# -------------------------------------------------------------
# 8️⃣ HEATMAPS
# -------------------------------------------------------------
fig, axes = plt.subplots(2, 2, figsize=(16, 14))

sns.heatmap(euclid_matrix, annot=True, fmt=".2f",
            cmap="plasma", ax=axes[0,0])
axes[0,0].set_title("Euclidean Distance")

sns.heatmap(manhattan_matrix, annot=True, fmt=".2f",
            cmap="cool", ax=axes[0,1])
axes[0,1].set_title("Manhattan Distance")

sns.heatmap(cosine_matrix, annot=True, fmt=".2f",
            cmap="viridis", ax=axes[1,0])
axes[1,0].set_title("Cosine Similarity")

sns.heatmap(pearson_matrix, annot=True, fmt=".2f",
            cmap="magma", center=0, ax=axes[1,1])
axes[1,1].set_title("Pearson Correlation")

plt.suptitle(
    "Cluster Similarity Analysis (GII Dataset)\nK-Means (k = 6)",
    fontsize=16,
    fontweight="bold"
)

plt.tight_layout()
plt.show()

print("\n✅ Cluster similarity analysis (k=6) completed successfully.")


In [ ]:
# =============================================================
# 🚀 K-MEANS (K=6) CLUSTER VALIDATION PIPELINE
# GII.csv — SSCI / Q1 Ready (+ CatBoost)
# =============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    confusion_matrix, classification_report, accuracy_score,
    silhouette_score, davies_bouldin_score, calinski_harabasz_score
)

# MODELS
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier

from catboost import CatBoostClassifier

# =====================================================
# 1️⃣ DATA LOAD
# =====================================================

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

# Country kolonunu ayır
country_col = "Country"

# Sayısal değişkenler (GII indicators)
features = df.select_dtypes(include=[np.number]).columns.tolist()
X_raw = df[features]

print("Veri boyutu:", df.shape)
print("Kullanılan değişkenler:", features)

# =====================================================
# 2️⃣ STANDARDIZATION
# =====================================================

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# =====================================================
# 3️⃣ K-MEANS (K = 6)
# =====================================================

k = 6
kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
y = kmeans.fit_predict(X)

df["Cluster"] = y

# =====================================================
# 4️⃣ INTERNAL CLUSTER VALIDATION
# =====================================================

print("\n" + "="*70)
print("📊 INTERNAL CLUSTER VALIDATION (GII, K=6)")
print("="*70)

print(f"Silhouette Score        : {silhouette_score(X, y):.4f}")
print(f"Davies–Bouldin Index    : {davies_bouldin_score(X, y):.4f}")
print(f"Calinski–Harabasz Index : {calinski_harabasz_score(X, y):.2f}")

# =====================================================
# 5️⃣ TRAIN–TEST SPLIT (PSEUDO-SUPERVISED VALIDATION)
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# =====================================================
# 6️⃣ CLASSIFIERS (WITH CATBOOST)
# =====================================================

models = {
    "Random Forest": RandomForestClassifier(n_estimators=300, random_state=42),

    "Logistic Regression": LogisticRegression(
        max_iter=1000, multi_class="multinomial", solver="lbfgs"
    ),

    "Decision Tree": DecisionTreeClassifier(random_state=42),

    "SVM (RBF)": SVC(kernel="rbf", C=1.5, gamma="scale"),

    "KNN": KNeighborsClassifier(n_neighbors=7, weights="distance"),

    "Naive Bayes": GaussianNB(),

    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=250,
        learning_rate=0.05,
        max_depth=3
    ),

    "ANN (MLP)": MLPClassifier(
        hidden_layer_sizes=(120, 60),
        max_iter=800,
        random_state=42
    ),

    "CatBoost": CatBoostClassifier(
        iterations=400,
        learning_rate=0.05,
        depth=6,
        loss_function="MultiClass",
        verbose=0,
        random_seed=42
    )
}

# =====================================================
# 7️⃣ CONFUSION MATRICES — 3x3 GRID
# =====================================================

fig, axes = plt.subplots(3, 3, figsize=(20, 18))
axes = axes.flatten()

for idx, (name, model) in enumerate(models.items()):

    print("\n" + "-"*70)
    print(f"▶ MODEL: {name}")
    print("-"*70)

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    acc = accuracy_score(y_test, y_pred)
    print(f"Accuracy: {acc:.4f}")
    print(classification_report(y_test, y_pred))

    cm = confusion_matrix(y_test, y_pred)

    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="viridis",
        xticklabels=[f"C{i+1}" for i in range(k)],
        yticklabels=[f"C{i+1}" for i in range(k)],
        ax=axes[idx]
    )

    axes[idx].set_title(f"{name}\nAcc = {acc:.3f}", fontweight="bold")
    axes[idx].set_xlabel("Predicted")
    axes[idx].set_ylabel("Actual")

# Fazla subplot varsa kapat
for j in range(idx + 1, len(axes)):
    axes[j].axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# =====================================================
# 3️⃣ WHICH CLUSTERS ARE HARDEST TO SEPARATE?
# Pseudo-supervised structural ambiguity analysis
# (Random Forest as near-optimal but imperfect separator)
# =====================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from sklearn.ensemble import RandomForestClassifier

# -----------------------------
# Train Random Forest
# -----------------------------
rf = RandomForestClassifier(
    n_estimators=300,
    random_state=42
)

rf.fit(X_train, y_train)
y_pred = rf.predict(X_test)

# -----------------------------
# Confusion Matrix
# -----------------------------
conf_mat = confusion_matrix(y_test, y_pred)
n_classes = conf_mat.shape[0]

conf_df = pd.DataFrame(
    conf_mat,
    index=[f"C{i+1}" for i in range(n_classes)],
    columns=[f"C{i+1}" for i in range(n_classes)]
)

# Normalize by true class (row-wise)
conf_norm = conf_df.div(conf_df.sum(axis=1), axis=0)

# -----------------------------
# Visualization
# -----------------------------
plt.figure(figsize=(9, 8))
sns.heatmap(
    conf_norm,
    annot=True,
    fmt=".2f",
    cmap="rocket_r",
    linewidths=1,
    cbar_kws={"label": "Classification Proportion"}
)

plt.title(
    "Cluster Confusion Structure:\nHard-to-Separate Innovation Regimes",
    fontsize=15,
    fontweight="bold"
)
plt.xlabel("Predicted Cluster")
plt.ylabel("True Cluster")
plt.tight_layout()
plt.show()

print(conf_norm)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score
)

# MODELS
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB
from sklearn.neural_network import MLPClassifier
from catboost import CatBoostClassifier

# =====================================================
# 🔧 VISUAL SETTINGS (PUBLICATION QUALITY)
# =====================================================

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.size": 13,
    "font.weight": "bold",
    "axes.labelweight": "bold",
    "axes.titleweight": "bold",
    "axes.linewidth": 1.6
})
sns.set_style("whitegrid")

# =====================================================
# 1️⃣ LOAD & PREPARE DATA
# =====================================================

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

features = df.select_dtypes(include=[np.number]).columns.tolist()
X_raw = df[features]

scaler = StandardScaler()
X = scaler.fit_transform(X_raw)

# =====================================================
# 2️⃣ K-MEANS (K = 6)
# =====================================================

k = 6
kmeans = KMeans(n_clusters=k, random_state=42, n_init=20)
y = kmeans.fit_predict(X)

# =====================================================
# 3️⃣ TRAIN / TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

# =====================================================
# 4️⃣ PARAMETER GRIDS
# =====================================================

param_grids = {

    "Random Forest": {
        "n_estimators": [200, 300],
        "max_depth": [None, 15]
    },

    "Logistic Regression": {
        "C": [0.1, 1, 10]
    },

    "Decision Tree": {
        "max_depth": [None, 10, 20]
    },

    "SVM (RBF)": {
        "C": [0.5, 1.5],
        "gamma": ["scale", "auto"]
    },

    "KNN": {
        "n_neighbors": [5, 7],
        "weights": ["distance"]
    },

    "Gaussian NB": {},

    "Gradient Boosting": {
        "n_estimators": [200, 300],
        "learning_rate": [0.05]
    },

    "ANN (MLP)": {
        "hidden_layer_sizes": [(120, 60)],
        "activation": ["relu"]
    },

    "CatBoost": {
        "iterations": [300, 400],
        "depth": [4, 6],
        "learning_rate": [0.05]
    }
}

# =====================================================
# 5️⃣ MODELS
# =====================================================

models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(
        max_iter=2000, multi_class="multinomial", solver="lbfgs"
    ),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "SVM (RBF)": SVC(kernel="rbf", probability=True, random_state=42),
    "KNN": KNeighborsClassifier(),
    "Gaussian NB": GaussianNB(),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    "ANN (MLP)": MLPClassifier(max_iter=1000, random_state=42),
    "CatBoost": CatBoostClassifier(
        verbose=0, loss_function="MultiClass", random_seed=42
    )
}

# =====================================================
# 6️⃣ GRIDSEARCH
# =====================================================

best_models = {}

print("\n" + "="*80)
print("🔍 GRIDSEARCH OPTIMIZATION (F1-MACRO)")
print("="*80)

for name, model in models.items():
    print(f"\n▶ {name}")

    grid = GridSearchCV(
        model,
        param_grids[name],
        cv=3,
        scoring="f1_macro",
        n_jobs=-1
    )

    grid.fit(X_train, y_train)
    best_models[name] = grid.best_estimator_

    print("  Best params:", grid.best_params_)
    print(f"  Best CV F1-macro: {grid.best_score_:.4f}")

# =====================================================
# 7️⃣ PERFORMANCE EVALUATION
# =====================================================

performance = []

for name, model in best_models.items():
    y_pred = model.predict(X_test)

    if hasattr(model, "predict_proba"):
        y_prob = model.predict_proba(X_test)
        auc = roc_auc_score(
            y_test, y_prob, multi_class="ovr", average="macro"
        )
    else:
        auc = np.nan

    performance.append([
        name,
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred, average="macro"),
        recall_score(y_test, y_pred, average="macro"),
        f1_score(y_test, y_pred, average="macro"),
        auc
    ])

perf_df = pd.DataFrame(
    performance,
    columns=["Model", "Accuracy", "Precision", "Recall", "F1-score", "Macro AUC"]
).set_index("Model")

print("\n" + "="*80)
print("📊 GRIDSEARCH PERFORMANCE TABLE")
print("="*80)
print(perf_df.round(4))

# =====================================================
# 8️⃣ BARPLOT — HIGH CONTRAST
# =====================================================

ax = perf_df.plot(
    kind="bar",
    figsize=(18, 9),
    width=0.8,
    colormap="viridis"
)

plt.title(
    "Model Performance After GridSearch Optimization\n(K-Means K=6 Cluster Validation)",
    fontsize=18,
    fontweight="bold"
)
plt.ylabel("Score")
plt.xticks(rotation=45, ha="right")
plt.ylim(0, 1.05)
plt.grid(axis="y", alpha=0.4)

for container in ax.containers:
    ax.bar_label(container, fmt="%.3f", fontsize=10, rotation=90)

plt.tight_layout()
plt.show()


In [ ]:
# ============================================================
# 1️⃣ KÜTÜPHANELER
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# Linear & Regularized
from sklearn.linear_model import (
    LinearRegression, Ridge, Lasso, ElasticNet,
    HuberRegressor, BayesianRidge
)

# Tree-based & Ensemble
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import (
    RandomForestRegressor, ExtraTreesRegressor,
    AdaBoostRegressor, GradientBoostingRegressor
)

# Other models
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.neural_network import MLPRegressor

# Boosting
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor
from catboost import CatBoostRegressor


# ============================================================
# 2️⃣ VERİ YÜKLEME
# ============================================================
df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

# ============================================================
# 🎯 HEDEF & ÖZELLİKLER
# ============================================================
target_col = "Score"

y = df[target_col]
X = df.drop(columns=["Country", target_col])
X = X.select_dtypes(include=[np.number])


# ============================================================
# 3️⃣ STANDARDİZASYON
# ============================================================
X_std = StandardScaler().fit_transform(X)
y_std = StandardScaler().fit_transform(
    y.values.reshape(-1, 1)
).ravel()


# ============================================================
# 4️⃣ TRAIN – TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_std, y_std,
    test_size=0.20,
    random_state=42
)


# ============================================================
# 5️⃣ REGRESYON MODELLERİ
# ============================================================
models = {
    "Huber": HuberRegressor(),
    "Bayesian Ridge": BayesianRidge(),
    "Linear Regression": LinearRegression(),
    "Ridge": Ridge(alpha=1.0),
    "Elastic Net": ElasticNet(alpha=0.05, l1_ratio=0.5),
    "Lasso": Lasso(alpha=0.05),

    "SVR (RBF)": SVR(C=10, epsilon=0.1),
    "k-NN": KNeighborsRegressor(n_neighbors=5),

    "MLP (ANN)": MLPRegressor(
        hidden_layer_sizes=(120, 60),
        max_iter=1000,
        random_state=42
    ),

    "Decision Tree": DecisionTreeRegressor(random_state=42),
    "Random Forest": RandomForestRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "Extra Trees": ExtraTreesRegressor(
        n_estimators=300, random_state=42, n_jobs=-1
    ),
    "AdaBoost": AdaBoostRegressor(
        n_estimators=300, learning_rate=0.05, random_state=42
    ),
    "Gradient Boosting": GradientBoostingRegressor(
        n_estimators=300, random_state=42
    ),

    "XGBoost": XGBRegressor(
        n_estimators=300,
        learning_rate=0.05,
        max_depth=6,
        random_state=42
    ),
    "LightGBM": LGBMRegressor(
        n_estimators=300,
        random_state=42
    ),
    "CatBoost": CatBoostRegressor(
        iterations=300,
        depth=4,
        learning_rate=0.05,
        verbose=0,
        random_state=42
    )
}


# ============================================================
# 6️⃣ EĞİTİM & PERFORMANS
# ============================================================
results = []

for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": name,
        "MSE": mean_squared_error(y_test, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R²": r2_score(y_test, y_pred)
    })

results_df = (
    pd.DataFrame(results)
    .sort_values(by="R²", ascending=False)
    .reset_index(drop=True)
)

print("\n================ AFFORDABILITY INDEX – ALL MODELS =================\n")
print(results_df.round(4))


# ============================================================
# 🎨 CANLI RENK PALETİ (SSCI-FIGURE READY)
# ============================================================
vivid_colors = [
    "#e41a1c", "#377eb8", "#4daf4a", "#984ea3",
    "#ff7f00", "#ffff33", "#a65628", "#f781bf",
    "#1b9e77", "#d95f02", "#7570b3", "#e7298a",
    "#66a61e", "#e6ab02", "#a6761d", "#666666",
    "#17becf"
]

model_colors = {
    model: vivid_colors[i % len(vivid_colors)]
    for i, model in enumerate(results_df["Model"])
}


# ============================================================
# 7️⃣ METRİK GÖRSELLEŞTİRME (2×2 PANEL)
# ============================================================
metrics = ["MSE", "RMSE", "MAE", "R²"]
titles = [
    "Mean Squared Error (MSE)",
    "Root Mean Squared Error (RMSE)",
    "Mean Absolute Error (MAE)",
    "R² Score"
]

fig, axes = plt.subplots(2, 2, figsize=(19, 18))
axes = axes.flatten()

for ax, metric, title in zip(axes, metrics, titles):
    bars = ax.bar(
        results_df["Model"],
        results_df[metric],
        color=[model_colors[m] for m in results_df["Model"]],
        edgecolor="black",
        linewidth=0.8
    )

    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2,
            h,
            f"{h:.4f}",
            ha="center",
            va="bottom",
            rotation=90,
            fontsize=9,
            fontweight="bold"
        )

    ax.set_title(title, fontsize=13, weight="bold")
    ax.set_xticklabels(results_df["Model"], rotation=45, ha="right")
    ax.grid(axis="y", linestyle="--", alpha=0.4)

    if metric == "R²":
        ax.set_ylim(0.70, 1.01)

plt.suptitle(
    "Comprehensive Performance Comparison of Regression Models\n(Target: Score)",
    fontsize=16,
    weight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()


In [ ]:
# ============================================================
# PUBLICATION-READY MODEL COMPARISON (LABELED VERSION)
# Consistent Feature Colors | Top-10 | Horizontal | Value Labels
# Extra Trees | CatBoost | Random Forest | Gradient Boosting
# ============================================================

# 1. LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor, GradientBoostingRegressor
from catboost import CatBoostRegressor

# 2. DATA LOADING
df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target_col = "Score"
y = df[target_col]
X = df.drop(columns=["Country", target_col])
X = X.select_dtypes(include=[np.number])
feature_labels = X.columns.tolist()

# 3. STANDARDIZATION
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_std = scaler_X.fit_transform(X)
y_std = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel()

# 4. TRAIN-TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X_std, y_std, test_size=0.20, random_state=42
)

# 5. MODELS
models = [
    ("Extra Trees", ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ("CatBoost", CatBoostRegressor(iterations=300, depth=4, learning_rate=0.05,
                                   verbose=0, random_state=42)),
    ("Random Forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ("Gradient Boosting", GradientBoostingRegressor(n_estimators=300, random_state=42))
]

# 6. GLOBAL COLOR MAPPING
unique_features = feature_labels
color_map = plt.cm.tab20(np.linspace(0, 1, len(unique_features)))
feature_color_dict = dict(zip(unique_features, color_map))

# 7. PUBLICATION STYLE SETTINGS
plt.rcParams.update({
    "font.size": 12,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.dpi": 300
})

# 8. FIGURE INITIALIZATION
fig, axes = plt.subplots(2, 2, figsize=(17, 13))
axes = axes.flatten()

performance_results = []
feature_importance_results = {}

# 9. MODEL LOOP
for ax, (name, model) in zip(axes, models):

    # Fit model
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    # Performance metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    performance_results.append({
        "Model": name,
        "R2": r2,
        "RMSE": rmse,
        "MAE": mae
    })

    # Feature importance
    fi = model.feature_importances_

    fi_df = pd.DataFrame({
        "Feature": feature_labels,
        "Importance": fi
    }).sort_values("Importance", ascending=False).head(10)

    feature_importance_results[name] = fi_df.copy()

    fi_df = fi_df.sort_values("Importance", ascending=True)
    colors = [feature_color_dict[f] for f in fi_df["Feature"]]

    # Horizontal bar chart
    bars = ax.barh(fi_df["Feature"], fi_df["Importance"], color=colors)

    # Value labels on bars
    for bar in bars:
        width = bar.get_width()
        ax.text(
            width + 0.002,
            bar.get_y() + bar.get_height() / 2,
            f"{width:.3f}",
            va="center",
            fontsize=14  # font büyüklüğü artırıldı
        )

    # Titles and labels
    ax.set_title(name, weight="bold", fontsize=14)
    ax.set_xlabel("Feature Importance", fontsize=14)
    ax.set_ylabel("Feature", fontsize=14)

    # Tick label fontunu büyüt
    ax.tick_params(axis='x', labelsize=12)
    ax.tick_params(axis='y', labelsize=12)

    ax.grid(axis="x", linestyle="--", alpha=0.3)

    # Performance metrics box
    ax.text(
        0.98, 0.05,
        f"R² = {r2:.3f}\nRMSE = {rmse:.4f}\nMAE = {mae:.4f}",
        transform=ax.transAxes,
        ha="right", va="bottom",
        fontsize=12,
        bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
    )

# 10. SUPERTITLE
plt.suptitle(
    "Feature Importance and Predictive Performance\nTree-Based Regression Models",
    fontsize=16,
    weight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# 11. PERFORMANCE TABLE
performance_df = pd.DataFrame(performance_results).sort_values("R2", ascending=False)

print("\n===== FINAL MODEL PERFORMANCE =====\n")
print(performance_df.round(4))

# 12. FEATURE IMPORTANCE TABLES
for model_name, fi_df in feature_importance_results.items():
    print(f"\n===== FEATURE IMPORTANCE: {model_name} =====\n")
    print(fi_df.round(4))

In [ ]:
# ============================================================
# FINAL MODEL COMPARISON – TOP 4 TREE-BASED REGRESSORS
# Gradient Boosting | Random Forest | Extra Trees | CatBoost
# Feature Importance & Performance (SSCI-Ready)
# ============================================================

# ============================================================
# 1️⃣ KÜTÜPHANELER
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import (
    GradientBoostingRegressor,
    RandomForestRegressor,
    ExtraTreesRegressor
)
from catboost import CatBoostRegressor

# ============================================================
# 2️⃣ VERİ YÜKLEME
# ============================================================
df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target_col = "Score"

y = df[target_col]
X = df.drop(columns=["Country", target_col])
X = X.select_dtypes(include=[np.number])

feature_labels = X.columns.tolist()

# ============================================================
# 3️⃣ STANDARDİZASYON
# ============================================================
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_std = scaler_X.fit_transform(X)
y_std = scaler_y.fit_transform(y.values.reshape(-1, 1)).ravel()

# ============================================================
# 4️⃣ TRAIN – TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_std, y_std, test_size=0.20, random_state=42
)

# ============================================================
# 5️⃣ EN İYİ 4 MODEL
# ============================================================
models = [
    ("Gradient Boosting", GradientBoostingRegressor(n_estimators=300, random_state=42)),
    ("Random Forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ("Extra Trees", ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ("CatBoost", CatBoostRegressor(iterations=300, depth=4, learning_rate=0.05,
                                   verbose=0, random_state=42))
]

# ============================================================
# 🎨 FEATURE RENK PALETİ
# ============================================================
tableau_colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
    "#bcbd22", "#17becf"
]

feature_colors = {
    f: tableau_colors[i % len(tableau_colors)]
    for i, f in enumerate(feature_labels)
}

# ============================================================
# 6️⃣ SONUÇ DEPOLARI
# ============================================================
performance_results = []
feature_importance_results = {}

# ============================================================
# 7️⃣ 2×2 PANEL GRAFİK – FEATURE IMPORTANCE
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(19, 14))
axes = axes.flatten()

for ax, (name, model) in zip(axes, models):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    performance_results.append({
        "Model": name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R²": r2
    })

    # 🌲 Tree-based Feature Importance
    fi = model.feature_importances_

    fi_df = pd.DataFrame({
        "Feature": feature_labels,
        "Importance": fi
    }).sort_values("Importance", ascending=False)

    feature_importance_results[name] = fi_df

    colors = [feature_colors[f] for f in fi_df["Feature"]]

    bars = ax.bar(
        fi_df["Feature"],
        fi_df["Importance"],
        color=colors,
        edgecolor="black",
        linewidth=0.9
    )

    ax.set_title(f"{name} – Feature Importance", fontsize=16, weight="bold")
    ax.set_ylabel("Relative Importance")
    ax.tick_params(axis="x", rotation=90)
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h,
                f"{h:.3f}", ha="center", va="bottom", fontsize=10, rotation=90)

    ax.text(
        0.98, 0.95,
        f"R²   = {r2:.3f}\nRMSE = {rmse:.4f}\nMAE  = {mae:.4f}",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=10,
        bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                  edgecolor="gray", alpha=0.95)
    )

# ============================================================
# 8️⃣ GENEL BAŞLIK
# ============================================================
plt.suptitle(
    "Final Model Comparison: Top-Performing Tree-Based Algorithms\n"
    "Feature Importance and Predictive Performance (Score)",
    fontsize=16,
    weight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# ============================================================
# 9️⃣ PERFORMANS TABLOSU
# ============================================================
performance_df = pd.DataFrame(performance_results).sort_values("R²", ascending=False)

print("\n================ FINAL MODEL PERFORMANCE (TOP 4) ================\n")
print(performance_df.round(5))

# ============================================================
# 🔟 FEATURE IMPORTANCE TABLOLARI
# ============================================================
for model_name, fi_df in feature_importance_results.items():
    print(f"\n================ FEATURE IMPORTANCE: {model_name} ================\n")
    print(fi_df.round(5))


In [ ]:
# ============================================================
# FINAL MODEL COMPARISON – TOP 4 TREE-BASED REGRESSORS
# Gradient Boosting | Random Forest | Extra Trees | CatBoost
# Feature Importance & Performance (SSCI-Ready)
# ============================================================

# ============================================================
# 1️⃣ LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from catboost import CatBoostRegressor

# ============================================================
# 2️⃣ LOAD DATA
# ============================================================
df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target_col = "Score"

y = df[target_col]
X = df.drop(columns=["Country", target_col], errors="ignore")
X = X.select_dtypes(include=[np.number])

feature_labels = X.columns.tolist()

# ============================================================
# 3️⃣ STANDARDIZATION (FOR FAIR COMPARISON)
# ============================================================
X_std = StandardScaler().fit_transform(X)
y_std = StandardScaler().fit_transform(y.values.reshape(-1, 1)).ravel()

# ============================================================
# 4️⃣ TRAIN – TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_std, y_std, test_size=0.20, random_state=42
)

# ============================================================
# 5️⃣ MODELS (TOP 4)
# ============================================================
models = [
    ("Gradient Boosting",
     GradientBoostingRegressor(n_estimators=300, random_state=42)),

    ("Random Forest",
     RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),

    ("Extra Trees",
     ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)),

    ("CatBoost",
     CatBoostRegressor(
         iterations=300,
         depth=4,
         learning_rate=0.05,
         random_state=42,
         verbose=0
     ))
]

# ============================================================
# 🎨 FEATURE COLOR PALETTE
# ============================================================
tableau_colors = [
    "#17becf", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
    "#bcbd22", "#1f77b4"
]

feature_colors = {
    f: tableau_colors[i % len(tableau_colors)]
    for i, f in enumerate(feature_labels)
}

# ============================================================
# 6️⃣ RESULT CONTAINERS
# ============================================================
performance_results = []
feature_importance_results = {}

# ============================================================
# 7️⃣ 2×2 PANEL – FEATURE IMPORTANCE
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(19, 14))
axes = axes.flatten()

for ax, (name, model) in zip(axes, models):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)

    performance_results.append({
        "Model": name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R²": r2
    })

    fi = model.feature_importances_

    fi_df = pd.DataFrame({
        "Feature": feature_labels,
        "Importance": fi
    }).sort_values("Importance", ascending=False)

    feature_importance_results[name] = fi_df

    colors = [feature_colors[f] for f in fi_df["Feature"]]

    bars = ax.bar(
        fi_df["Feature"],
        fi_df["Importance"],
        color=colors,
        edgecolor="black",
        linewidth=0.9
    )

    ax.set_title(f"{name} – Feature Importance", fontsize=13, weight="bold")
    ax.set_ylabel("Relative Importance")
    ax.tick_params(axis="x", rotation=90)
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    for bar in bars:
        h = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            h,
            f"{h:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
            rotation=90
        )

    ax.text(
        0.98, 0.95,
        f"R²   = {r2:.3f}\nRMSE = {rmse:.4f}\nMAE  = {mae:.4f}",
        transform=ax.transAxes,
        ha="right",
        va="top",
        fontsize=10,
        bbox=dict(
            boxstyle="round,pad=0.4",
            facecolor="white",
            edgecolor="gray",
            alpha=0.95
        )
    )

# ============================================================
# 8️⃣ GLOBAL TITLE
# ============================================================
plt.suptitle(
    "Final Model Comparison: Top-Performing Tree-Based Algorithms\n"
    "Feature Importance and Predictive Performance (Score)",
    fontsize=16,
    weight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# ============================================================
# 9️⃣ PERFORMANCE TABLE
# ============================================================
performance_df = (
    pd.DataFrame(performance_results)
    .sort_values("R²", ascending=False)
)

print("\n================ FINAL MODEL PERFORMANCE (TOP 4) ================\n")
print(performance_df.round(5))

# ============================================================
# 🔟 FEATURE IMPORTANCE TABLES
# ============================================================
for model_name, fi_df in feature_importance_results.items():
    print(f"\n================ FEATURE IMPORTANCE: {model_name} ================\n")
    print(fi_df.round(5))


In [ ]:
# ============================================================
# FINAL MODEL COMPARISON – TREE-BASED REGRESSORS (PPI)
# Gradient Boosting | Random Forest | Extra Trees | CatBoost
# Feature Importance & Model Performance (SSCI-Ready)
# ============================================================

# ============================================================
# 1️⃣ LIBRARIES
# ============================================================
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor, ExtraTreesRegressor
from catboost import CatBoostRegressor

# ============================================================
# 2️⃣ LOAD DATA
# ============================================================
df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target_col = "Score"

y = df[target_col]
X = df.drop(columns=["Country", target_col], errors="ignore")
X = X.select_dtypes(include=[np.number])

feature_labels = X.columns.tolist()

# ============================================================
# 3️⃣ STANDARDIZATION
# ============================================================
X_std = StandardScaler().fit_transform(X)
y_std = StandardScaler().fit_transform(y.values.reshape(-1, 1)).ravel()

# ============================================================
# 4️⃣ TRAIN – TEST SPLIT
# ============================================================
X_train, X_test, y_train, y_test = train_test_split(
    X_std, y_std, test_size=0.20, random_state=42
)

# ============================================================
# 5️⃣ MODELS
# ============================================================
models = [
    ("Gradient Boosting", GradientBoostingRegressor(n_estimators=300, random_state=42)),
    ("Random Forest", RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ("Extra Trees", ExtraTreesRegressor(n_estimators=300, random_state=42, n_jobs=-1)),
    ("CatBoost", CatBoostRegressor(iterations=300, depth=4,
                                   learning_rate=0.05,
                                   random_state=42, verbose=0))
]

# ============================================================
# 🎨 FEATURE COLOR PALETTE (HAPI STYLE)
# ============================================================
tableau_colors = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728",
    "#9467bd", "#8c564b", "#e377c2", "#7f7f7f",
    "#bcbd22", "#17becf"
]

feature_colors = {
    f: tableau_colors[i % len(tableau_colors)]
    for i, f in enumerate(feature_labels)
}

# ============================================================
# 6️⃣ RESULT CONTAINERS
# ============================================================
performance_results = []
feature_importance_results = {}

# ============================================================
# 7️⃣ PANEL GRAPH (2×2) – HAPI FORMAT
# ============================================================
fig, axes = plt.subplots(2, 2, figsize=(18, 13), sharex=True)
axes = axes.flatten()

for ax, (name, model) in zip(axes, models):

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae  = mean_absolute_error(y_test, y_pred)
    r2   = r2_score(y_test, y_pred)

    performance_results.append({
        "Model": name,
        "MSE": mse,
        "RMSE": rmse,
        "MAE": mae,
        "R²": r2
    })

    fi = model.feature_importances_

    fi_df = pd.DataFrame({
        "Feature": feature_labels,
        "Importance": fi
    }).sort_values("Importance", ascending=False)

    feature_importance_results[name] = fi_df

    colors = [feature_colors[f] for f in fi_df["Feature"]]

    bars = ax.bar(
        fi_df["Feature"],
        fi_df["Importance"],
        color=colors,
        edgecolor="black",
        linewidth=0.95
    )

    ax.set_title(f"{name}", fontsize=17, weight="bold")
    ax.set_ylabel("Relative Feature Importance")
    ax.tick_params(axis="x", rotation=90,)
    ax.grid(axis="y", linestyle="--", alpha=0.35)

    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2,
                h, f"{h:.3f}",
                ha="center", va="bottom",
                fontsize=11, rotation=90)

    ax.text(
        0.98, 0.95,
        f"R²   = {r2:.3f}\nRMSE = {rmse:.4f}\nMAE  = {mae:.4f}",
        transform=ax.transAxes,
        ha="right", va="top",
        fontsize=16,
        bbox=dict(boxstyle="round,pad=0.4",
                  facecolor="white",
                  edgecolor="gray",
                  alpha=0.95)
    )

    legend_patches = [
        mpatches.Patch(color=feature_colors[f], label=f)
        for f in fi_df["Feature"]
    ]
    ax.legend(handles=legend_patches, fontsize=10,
              loc="lower left", frameon=True)

# ============================================================
# 8️⃣ GLOBAL TITLE
# ============================================================
plt.suptitle(
    "Final Model Comparison: Tree-Based Algorithms\n"
    "Feature Importance and Predictive Performance (Affordability Index)",
    fontsize=16, weight="bold"
)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# ============================================================
# 9️⃣ PERFORMANCE TABLE
# ============================================================
performance_df = pd.DataFrame(performance_results).sort_values("R²", ascending=False)

print("\n================ FINAL MODEL PERFORMANCE =================\n")
print(performance_df.round(5))

# ============================================================
# 🔟 FEATURE IMPORTANCE TABLES
# ============================================================
for model_name, fi_df in feature_importance_results.items():
    print(f"\n================ FEATURE IMPORTANCE: {model_name} ================\n")
    print(fi_df.round(5))


In [ ]:
df

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

# -----------------------------------------------------
# 1️⃣ LOAD DATA
# -----------------------------------------------------

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target_col = "Score"

y = df[target_col]

X = (
    df.drop(columns=["Country", target_col], errors="ignore")
      .select_dtypes(include=[np.number])
)

feature_labels = X.columns.tolist()

# -----------------------------------------------------
# 2️⃣ STANDARDIZATION
# -----------------------------------------------------

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_std = scaler_X.fit_transform(X)
y_std = scaler_y.fit_transform(y.values.reshape(-1,1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X_std, y_std, test_size=0.20, random_state=42
)

# -----------------------------------------------------
# 3️⃣ EXTRA TREES MODEL
# -----------------------------------------------------

et_model = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

et_model.fit(X_train, y_train)
y_pred = et_model.predict(X_test)

# -----------------------------------------------------
# 4️⃣ PERFORMANCE METRICS
# -----------------------------------------------------

metrics = {
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "MAE": mean_absolute_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred)
}

print("\n=========== EXTRA TREES PERFORMANCE ===========\n")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# -----------------------------------------------------
# 5️⃣ SHAP TREE EXPLAINER
# -----------------------------------------------------

explainer = shap.TreeExplainer(et_model)
shap_values = explainer.shap_values(X_test)

# -----------------------------------------------------
# 6️⃣ SHAP SUMMARY PLOT (GLOBAL + LOCAL)
# -----------------------------------------------------

shap.summary_plot(
    shap_values,
    X_test,
    feature_names=feature_labels,
    show=True
)

# -----------------------------------------------------
# 7️⃣ MEAN |SHAP| (GLOBAL IMPORTANCE)
# -----------------------------------------------------

shap_mean = np.abs(shap_values).mean(axis=0)

shap_df = (
    pd.DataFrame({
        "Feature": feature_labels,
        "Mean |SHAP|": shap_mean
    })
    .sort_values("Mean |SHAP|", ascending=False)
)

print("\n=========== MEAN |SHAP| VALUES ===========\n")
print(shap_df.round(4))

# -----------------------------------------------------
# 8️⃣ BARPLOT – SSCI STYLE (WITH PERFORMANCE BOX)
# -----------------------------------------------------

plt.figure(figsize=(10,6))

sns.barplot(
    x="Mean |SHAP|",
    y="Feature",
    data=shap_df,
    palette="viridis"
)

plt.title("Global Feature Importance (Mean |SHAP|)\nExtra Trees – GII Score")
plt.xlabel("Mean |SHAP| Contribution")
plt.ylabel("Feature")

# Value labels on bars
for i, v in enumerate(shap_df["Mean |SHAP|"]):
    plt.text(v, i, f"{v:.3f}", va="center")

# Performance metrics box
performance_text = (
    f"MSE: {metrics['MSE']:.4f}\n"
    f"RMSE: {metrics['RMSE']:.4f}\n"
    f"MAE: {metrics['MAE']:.4f}\n"
    f"R²: {metrics['R2']:.4f}"
)

plt.gca().text(
    0.98, 0.05,
    performance_text,
    transform=plt.gca().transAxes,
    ha="right",
    va="bottom",
    fontsize=13,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.linear_model import Ridge

# -----------------------------------------------------
# 1️⃣ LOAD DATA
# -----------------------------------------------------
df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target_col = "Score"

y = df[target_col]
X = df.drop(columns=["Country", target_col], errors="ignore") \
      .select_dtypes(include=[np.number])

feature_labels = X.columns.tolist()

# -----------------------------------------------------
# 2️⃣ STANDARDIZATION
# -----------------------------------------------------
scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_std = scaler_X.fit_transform(X)
y_std = scaler_y.fit_transform(y.values.reshape(-1,1)).ravel()

X_train, X_test, y_train, y_test = train_test_split(
    X_std, y_std, test_size=0.20, random_state=42
)

# -----------------------------------------------------
# 3️⃣ EXTRA TREES MODEL
# -----------------------------------------------------
et_model = ExtraTreesRegressor(
    n_estimators=300,
    random_state=42,
    n_jobs=-1
)

et_model.fit(X_train, y_train)
y_pred = et_model.predict(X_test)

# -----------------------------------------------------
# 4️⃣ PERFORMANCE METRICS
# -----------------------------------------------------
metrics = {
    "MSE": mean_squared_error(y_test, y_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
    "MAE": mean_absolute_error(y_test, y_pred),
    "R2": r2_score(y_test, y_pred)
}

print("\n=========== EXTRA TREES PERFORMANCE ===========\n")
for k, v in metrics.items():
    print(f"{k}: {v:.4f}")

# -----------------------------------------------------
# 5️⃣ SHAP EXPLAINER
# -----------------------------------------------------
explainer = shap.TreeExplainer(et_model)
shap_values = explainer.shap_values(X_test)

shap_mean = np.abs(shap_values).mean(axis=0)

shap_df = pd.DataFrame({
    "Feature": feature_labels,
    "Mean |SHAP|": shap_mean
}).sort_values("Mean |SHAP|", ascending=False)

# -----------------------------------------------------
# 6️⃣ GLOBAL SHAP + PERFORMANCE METRICS
# -----------------------------------------------------
plt.figure(figsize=(10,6))
sns.barplot(
    x="Mean |SHAP|",
    y="Feature",
    data=shap_df,
    palette="viridis"
)

for i, v in enumerate(shap_df["Mean |SHAP|"]):
    plt.text(v, i, f"{v:.3f}", va="center")

perf_text = (
    f"MSE: {metrics['MSE']:.4f}\n"
    f"RMSE: {metrics['RMSE']:.4f}\n"
    f"MAE: {metrics['MAE']:.4f}\n"
    f"R²: {metrics['R2']:.4f}"
)

plt.gca().text(
    0.98, 0.05,
    perf_text,
    transform=plt.gca().transAxes,
    ha="right",
    va="bottom",
    fontsize=11,
    bbox=dict(boxstyle="round", facecolor="white", alpha=0.9)
)

plt.title("Global Feature Importance with Performance Metrics")
plt.tight_layout()
plt.show()

# -----------------------------------------------------
# 7️⃣ SHAP DEPENDENCE PLOTS (2x4 GRID)
# -----------------------------------------------------
top_n = min(8, len(feature_labels))
top_features = shap_df["Feature"].iloc[:top_n].values

fig, axes = plt.subplots(4, 2, figsize=(12, 20))
axes = axes.flatten()

for i, feature in enumerate(top_features):
    feature_index = feature_labels.index(feature)

    shap.dependence_plot(
        feature_index,
        shap_values,
        X_test,
        feature_names=feature_labels,
        show=False,
        ax=axes[i]
    )

    axes[i].set_title(feature, fontsize=10)

for j in range(i+1, 8):
    fig.delaxes(axes[j])

plt.suptitle("SHAP Dependence Plots (Top 8 Features)\nExtra Trees – GII Score",
             fontsize=14)

plt.tight_layout(rect=[0, 0, 1, 0.95])
plt.show()

# -----------------------------------------------------
# 8️⃣ SHAP INTERACTION MATRIX
# -----------------------------------------------------
shap_interaction_values = explainer.shap_interaction_values(X_test)
interaction_mean = np.abs(shap_interaction_values).mean(axis=0)

interaction_df = pd.DataFrame(
    interaction_mean,
    index=feature_labels,
    columns=feature_labels
)

print("\n=========== SHAP INTERACTION MATRIX ===========\n")
print(interaction_df.round(4))

plt.figure(figsize=(12,10))
sns.heatmap(
    interaction_df,
    cmap="viridis",
    annot=True,
    fmt=".3f"
)

plt.title("SHAP Interaction Matrix – Extra Trees")
plt.tight_layout()
plt.show()

# -----------------------------------------------------
# 9️⃣ LINEAR (RIDGE) vs SHAP COMPARISON (ETİKETLİ)
# -----------------------------------------------------
ridge_model = Ridge()
ridge_model.fit(X_train, y_train)

ridge_coef = np.abs(ridge_model.coef_)

linear_vs_shap_df = pd.DataFrame({
    "Feature": feature_labels,
    "Linear (Ridge) Coef": ridge_coef,
    "SHAP Mean |Value|": shap_mean
}).sort_values("SHAP Mean |Value|", ascending=False)

print("\n=========== LINEAR vs SHAP COMPARISON ===========\n")
print(linear_vs_shap_df.round(4))

plt.figure(figsize=(10,6))

sns.scatterplot(
    x="Linear (Ridge) Coef",
    y="SHAP Mean |Value|",
    hue="Feature",
    data=linear_vs_shap_df,
    palette="tab10",
    s=140,
    legend=False
)

# Katsayıları grafik üzerinde yaz
for i, row in linear_vs_shap_df.iterrows():
    label_text = (
        f"{row['Feature']}\n"
        f"L: {row['Linear (Ridge) Coef']:.3f} | "
        f"S: {row['SHAP Mean |Value|']:.3f}"
    )

    plt.text(
        row["Linear (Ridge) Coef"] + 0.005,
        row["SHAP Mean |Value|"] + 0.005,
        label_text,
        fontsize=8
    )

plt.xlabel("Linear (Ridge) Absolute Coefficients")
plt.ylabel("SHAP Mean |Value|")
plt.title("Linear vs SHAP Feature Importance Comparison")
plt.tight_layout()
plt.show()

# -----------------------------------------------------
# 🔟 EXTRA TREES ±1 SD SCENARIO ANALYSIS
# -----------------------------------------------------

print("\n=========== EXTRA TREES ±1 SD SCENARIO ANALYSIS ===========\n")

# Test verisini DataFrame yap
X_test_df = pd.DataFrame(X_test, columns=feature_labels)

scenario_results = []

for f in feature_labels:
    sd = X_test_df[f].std()

    X_plus = X_test_df.copy()
    X_minus = X_test_df.copy()

    # +1 SD ve -1 SD senaryosu
    X_plus[f] += sd
    X_minus[f] -= sd

    shap_plus = explainer.shap_values(X_plus)
    shap_minus = explainer.shap_values(X_minus)

    idx = feature_labels.index(f)

    scenario_results.append({
        "Feature": f,
        "+1 SD Effect": np.mean(shap_plus[:, idx] - shap_values[:, idx]),
        "-1 SD Effect": np.mean(shap_minus[:, idx] - shap_values[:, idx])
    })

scenario_df = pd.DataFrame(scenario_results).sort_values(
    by="+1 SD Effect", key=np.abs, ascending=False
)

print(scenario_df.round(4))


# -----------------------------------------------------
# SCENARIO GRAFİĞİ
# -----------------------------------------------------

scenario_melt = scenario_df.melt(
    id_vars="Feature",
    value_vars=["+1 SD Effect", "-1 SD Effect"],
    var_name="Scenario",
    value_name="Effect"
)

plt.figure(figsize=(10,6))
ax2 = sns.barplot(
    x="Effect",
    y="Feature",
    hue="Scenario",
    data=scenario_melt,
    palette="coolwarm"
)

plt.title("±1 SD Scenario Effects on GII Score (Extra Trees)")
plt.xlabel("Change in Predicted GII (Std Units)")
plt.ylabel("Feature")
plt.legend(title="")

# Sayısal değerleri çubuk üzerine yaz
for p in ax2.patches:
    w = p.get_width()
    ax2.text(
        w + 0.01*np.sign(w),
        p.get_y() + p.get_height()/2,
        f"{w:.3f}",
        ha="left" if w > 0 else "right",
        va="center"
    )

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error

# =====================================================
# 1. LOAD DATA
# =====================================================

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target = "Score"

df = df.select_dtypes(include=[np.number]).dropna()

features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

# =====================================================
# 2. TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =====================================================
# 3. MODEL
# =====================================================

model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

print("\nMODEL PERFORMANCE")
print("R²:", round(r2_score(y_test, y_pred), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))

In [ ]:
import numpy as np
import pandas as pd
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr, kendalltau

# =====================================================
# GLOBAL STYLE
# =====================================================

mpl.rcParams.update({
    "figure.titlesize": 12,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "font.size": 10
})

# =====================================================
# 1. LOAD DATA
# =====================================================

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target = "Score"

df = df.select_dtypes(include=[np.number]).dropna()

features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

# =====================================================
# 2. TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=features)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=features)

# =====================================================
# 3. MODEL
# =====================================================

model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled_df, y_train)
y_pred = model.predict(X_test_scaled_df)

print("\nMODEL PERFORMANCE")
print("R²:", round(r2_score(y_test, y_pred), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))

# =====================================================
# 4. SHAP GLOBAL IMPORTANCE
# =====================================================

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled_df)

shap_importance = np.abs(shap_values).mean(axis=0)

shap_ranking = pd.Series(
    shap_importance,
    index=features
).sort_values(ascending=False)

print("\nTOP 10 SHAP FEATURES")
print(shap_ranking.head(10))

# =====================================================
# 5. LIME LOCAL IMPORTANCE
# =====================================================

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=features,
    mode="regression",
    discretize_continuous=True,
    random_state=42
)

test_df = X_test_scaled_df.copy()
test_df["Prediction"] = y_pred
test_df["Actual"] = y_test.values

cases = {
    "Low Prediction": test_df["Prediction"].idxmin(),
    "Median Prediction": test_df["Prediction"].sub(
        test_df["Prediction"].median()
    ).abs().idxmin(),
    "High Prediction": test_df["Prediction"].idxmax()
}

# ===============================
# PRINT NUMERIC VALUES OF CASES
# ===============================

print("\n==============================")
print("CASE NUMERIC VALUES")
print("==============================")

for label, idx in cases.items():
    print(f"\n{label}")
    print("Index:", idx)
    print("Predicted Score:", round(test_df.loc[idx, "Prediction"], 4))
    print("Actual Score   :", round(test_df.loc[idx, "Actual"], 4))

# ===============================
# LIME COMPUTATION
# ===============================

fig, axes = plt.subplots(1, 3, figsize=(14, 4))
color_map = plt.cm.tab20.colors
lime_importance_global = np.zeros(len(features))

for ax, (label, idx) in zip(axes, cases.items()):

    exp = explainer_lime.explain_instance(
        X_test_scaled[idx],
        model.predict,
        num_features=8
    )

    df_plot = pd.DataFrame(
        exp.as_list(),
        columns=["Feature", "Contribution"]
    ).sort_values("Contribution")

    colors = [color_map[i % len(color_map)] for i in range(len(df_plot))]

    ax.barh(
        df_plot["Feature"],
        df_plot["Contribution"],
        color=colors
    )

    ax.set_title(f"{label}\n(Local R² = {exp.score:.3f})")

    for feat, val in exp.as_list():
        fname = feat.split(" ")[0]
        if fname in features:
            lime_importance_global[features.index(fname)] += abs(val)

fig.suptitle("LIME Local Explanations")
plt.tight_layout()
plt.show()

lime_importance_global /= 3

lime_ranking = pd.Series(
    lime_importance_global,
    index=features
).sort_values(ascending=False)

# =====================================================
# 6. SHAP – LIME CONSISTENCY
# =====================================================

common = list(set(shap_ranking.index) & set(lime_ranking.index))

shap_rank = shap_ranking.loc[common].rank()
lime_rank = lime_ranking.loc[common].rank()

print("\nCONSISTENCY RESULTS")
print("Spearman:", spearmanr(shap_rank, lime_rank))
print("Kendall :", kendalltau(shap_rank, lime_rank))

# =====================================================
# 7. LABELED NUMERIC OUTPUT
# =====================================================

print("\n==============================")
print("LABELED SHAP VALUES")
print("==============================")

shap_df = pd.DataFrame({
    "Feature": features,
    "SHAP Importance": shap_importance
}).sort_values("SHAP Importance", ascending=False)

print(shap_df)

print("\n==============================")
print("LABELED LIME VALUES")
print("==============================")

lime_df = pd.DataFrame({
    "Feature": features,
    "LIME Importance": lime_importance_global
}).sort_values("LIME Importance", ascending=False)

print(lime_df)

print("\n==============================")
print("SHAP & LIME MERGED TABLE")
print("==============================")

merged_df = shap_df.merge(lime_df, on="Feature", how="inner")
print(merged_df)

print("\nPIPELINE COMPLETED SUCCESSFULLY")

In [ ]:
import numpy as np
import pandas as pd
import shap
import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import matplotlib as mpl

from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score, mean_absolute_error
from scipy.stats import spearmanr, kendalltau

# =====================================================
# GLOBAL STYLE
# =====================================================

mpl.rcParams.update({
    "figure.titlesize": 12,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "font.size": 10
})

# =====================================================
# 1. LOAD DATA
# =====================================================

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target = "Score"
df = df.select_dtypes(include=[np.number]).dropna()

features = [c for c in df.columns if c != target]

X = df[features]
y = df[target]

# =====================================================
# 2. TRAIN TEST SPLIT
# =====================================================

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=features)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=features)

# =====================================================
# 3. MODEL
# =====================================================

model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_scaled_df, y_train)
y_pred = model.predict(X_test_scaled_df)

print("\nMODEL PERFORMANCE")
print("R² :", round(r2_score(y_test, y_pred), 4))
print("MAE:", round(mean_absolute_error(y_test, y_pred), 4))

# =====================================================
# 4. SHAP GLOBAL IMPORTANCE
# =====================================================

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_train_scaled_df)

shap_importance = np.abs(shap_values).mean(axis=0)

shap_ranking = pd.Series(
    shap_importance,
    index=features
).sort_values(ascending=False)

print("\nSHAP FEATURE IMPORTANCE (ALL VARIABLES)")
print(shap_ranking)

# =====================================================
# 5. LIME LOCAL + GLOBAL IMPORTANCE
# =====================================================

explainer_lime = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=features,
    mode="regression",
    discretize_continuous=False,
    random_state=42
)

lime_importance_global = np.zeros(len(features))

# Daha sağlam global LIME: tüm test setinden ortalama
for i in range(len(X_test_scaled_df)):

    exp = explainer_lime.explain_instance(
        X_test_scaled[i],
        model.predict,
        num_features=len(features)
    )

    for feat, val in exp.as_list():

        # feature ismi tam eşleşme için düzeltme
        clean_feat = feat.split("=")[0].strip()

        if clean_feat in features:
            lime_importance_global[features.index(clean_feat)] += abs(val)

lime_importance_global /= len(X_test_scaled_df)

lime_ranking = pd.Series(
    lime_importance_global,
    index=features
).sort_values(ascending=False)

print("\nLIME GLOBAL IMPORTANCE (ALL VARIABLES)")
print(lime_ranking)

# =====================================================
# 6. CONSISTENCY ANALYSIS
# =====================================================

common = list(set(shap_ranking.index) & set(lime_ranking.index))

shap_rank = shap_ranking.loc[common].rank()
lime_rank = lime_ranking.loc[common].rank()

print("\nCONSISTENCY RESULTS")
print("Spearman:", spearmanr(shap_rank, lime_rank))
print("Kendall :", kendalltau(shap_rank, lime_rank))

# =====================================================
# 7. MERGED TABLE
# =====================================================

merged_df = pd.DataFrame({
    "Feature": features,
    "SHAP Importance": shap_importance,
    "LIME Importance": lime_importance_global
}).sort_values("SHAP Importance", ascending=False)

print("\nSHAP & LIME MERGED TABLE")
print(merged_df)

print("\nPIPELINE COMPLETED SUCCESSFULLY")

# =====================================================
# 8. VISUALIZATION SECTION
# =====================================================

import seaborn as sns

sns.set_style("whitegrid")

# -------------------------------
# SHAP vs LIME Comparison Plot
# -------------------------------

plot_df = merged_df.copy()

fig, ax = plt.subplots(figsize=(10, 6))

bar_width = 0.35
x = np.arange(len(plot_df))

bars1 = ax.bar(
    x - bar_width/2,
    plot_df["SHAP Importance"],
    width=bar_width,
    label="SHAP",
    color="#1f77b4"
)

bars2 = ax.bar(
    x + bar_width/2,
    plot_df["LIME Importance"],
    width=bar_width,
    label="LIME",
    color="#ff7f0e"
)

ax.set_xticks(x)
ax.set_xticklabels(plot_df["Feature"], rotation=45, ha="right")
ax.set_title("SHAP vs LIME Feature Importance")
ax.set_ylabel("Importance Value")
ax.legend()

# Value labels on bars
for bars in [bars1, bars2]:
    for bar in bars:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width()/2,
            height,
            f"{height:.2f}",
            ha='center',
            va='bottom',
            fontsize=8,
            fontweight="bold"
        )

plt.tight_layout()
plt.show()

# -------------------------------
# Correlation Heatmap
# -------------------------------

corr_df = pd.DataFrame({
    "SHAP": shap_importance,
    "LIME": lime_importance_global
})

fig, ax = plt.subplots(figsize=(6, 5))

sns.heatmap(
    corr_df.corr(),
    annot=True,
    cmap="coolwarm",
    fmt=".3f",
    linewidths=0.5,
    ax=ax
)

ax.set_title("Correlation Between SHAP and LIME")
plt.tight_layout()
plt.show()

# -------------------------------
# Model Performance Visualization
# -------------------------------

metrics_df = pd.DataFrame({
    "Metric": ["R²", "MAE"],
    "Value": [r2_score(y_test, y_pred), mean_absolute_error(y_test, y_pred)]
})

fig, ax = plt.subplots(figsize=(6, 4))

bars = ax.bar(
    metrics_df["Metric"],
    metrics_df["Value"],
    color=["#2ca02c", "#d62728"]
)

ax.set_title("Model Performance Metrics")

for bar in bars:
    height = bar.get_height()
    ax.text(
        bar.get_x() + bar.get_width()/2,
        height,
        f"{height:.3f}",
        ha='center',
        va='bottom',
        fontsize=9,
        fontweight="bold"
    )

plt.tight_layout()
plt.show()

print("\nVISUALIZATION COMPLETED SUCCESSFULLY")

ax.text(
    0.5, 0.25,
    textstr,
    transform=ax.transAxes,
    fontsize=10,
    ha='center',
    verticalalignment='bottom',
    bbox=props
)

In [ ]:
# =====================================================
# 🔥 SHAP–LIME RANK CONSISTENCY ANALYSIS (FULL FIXED)
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import spearmanr

sns.set_style("whitegrid")

# --------------------------------------------------
# 0️⃣ MODEL TRAIN
# --------------------------------------------------

model = ExtraTreesRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

# --------------------------------------------------
# 1️⃣ SHAP VALUES
# --------------------------------------------------

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap_importance = np.abs(shap_values).mean(axis=0)

feature_labels = list(X_train.columns)

shap_rank_df = pd.DataFrame({
    "Feature": feature_labels,
    "SHAP_Importance": shap_importance
}).sort_values("SHAP_Importance", ascending=False)

shap_rank_df["SHAP_Rank"] = range(1, len(shap_rank_df) + 1)

print("\n=========== SHAP IMPORTANCE ===========")
print(shap_rank_df.round(4))

# --------------------------------------------------
# 2️⃣ LIME VALUES
# --------------------------------------------------

X_train_np = X_train.values
X_test_np = X_test.values

lime_explainer = LimeTabularExplainer(
    X_train_np,
    feature_names=feature_labels,
    mode="regression"
)

lime_importance = np.zeros(len(feature_labels))

n_samples = min(50, len(X_test_np))

for i in range(n_samples):
    exp = lime_explainer.explain_instance(
        X_test_np[i],
        model.predict,   # 🔥 artık tanımlı
        num_features=len(feature_labels)
    )

    for feat, val in exp.as_list():
        clean_feat = feat.split(" ")[0]
        if clean_feat in feature_labels:
            idx = feature_labels.index(clean_feat)
            lime_importance[idx] += abs(val)

lime_importance /= n_samples

lime_rank_df = pd.DataFrame({
    "Feature": feature_labels,
    "LIME_Importance": lime_importance
}).sort_values("LIME_Importance", ascending=False)

lime_rank_df["LIME_Rank"] = range(1, len(lime_rank_df) + 1)

print("\n=========== LIME IMPORTANCE ===========")
print(lime_rank_df.round(4))

# --------------------------------------------------
# 3️⃣ MERGE & RANK DIFFERENCE
# --------------------------------------------------

plot_df = shap_rank_df.merge(lime_rank_df, on="Feature")
plot_df["Rank_Diff"] = abs(plot_df["SHAP_Rank"] - plot_df["LIME_Rank"])
rank_df = plot_df.sort_values("Rank_Diff")

print("\n=========== RANK DIFFERENCE ===========")
print(rank_df[["Feature", "SHAP_Rank", "LIME_Rank", "Rank_Diff"]])

# --------------------------------------------------
# 4️⃣ VISUALIZATION
# --------------------------------------------------

plt.figure(figsize=(10, 6))

colors = plt.cm.viridis(np.linspace(0, 1, len(rank_df)))

bars = plt.barh(
    rank_df["Feature"],
    rank_df["Rank_Diff"],
    color=colors,
    edgecolor="black"
)

plt.title(
    "SHAP–LIME Rank Difference\n(Lower = Higher Consistency)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Absolute Rank Difference", fontsize=12)
plt.ylabel("Feature", fontsize=12)

for bar in bars:
    width = bar.get_width()
    plt.text(
        width + 0.05,
        bar.get_y() + bar.get_height()/2,
        f"{width:.0f}",
        va="center",
        fontsize=10
    )

plt.tight_layout()
plt.show()

# --------------------------------------------------
# 5️⃣ SPEARMAN CONSISTENCY SCORE
# --------------------------------------------------

rho, pval = spearmanr(plot_df["SHAP_Rank"], plot_df["LIME_Rank"])

print("\n=========== RANK CONSISTENCY ===========")
print(f"Spearman ρ: {rho:.3f}")
print(f"p-value: {pval:.4f}")

In [ ]:
# =====================================================
# 🔥 SHAP–LIME RANK CONSISTENCY ANALYSIS (FULL WORKING)
# =====================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import shap
from lime.lime_tabular import LimeTabularExplainer
from sklearn.ensemble import ExtraTreesRegressor
from scipy.stats import spearmanr

sns.set_style("whitegrid")

# --------------------------------------------------
# 0️⃣ MODEL TRAIN
# --------------------------------------------------

model = ExtraTreesRegressor(
    n_estimators=400,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

feature_labels = list(X_train.columns)

# --------------------------------------------------
# 1️⃣ SHAP VALUES
# --------------------------------------------------

explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

shap_importance = np.abs(shap_values).mean(axis=0)

shap_rank_df = pd.DataFrame({
    "Feature": feature_labels,
    "SHAP_Importance": shap_importance
}).sort_values("SHAP_Importance", ascending=False)

shap_rank_df["SHAP_Rank"] = range(1, len(shap_rank_df) + 1)

print("\n=========== SHAP IMPORTANCE ===========")
print(shap_rank_df.round(4))

# --------------------------------------------------
# 2️⃣ LIME VALUES
# --------------------------------------------------

X_train_np = X_train.values
X_test_np = X_test.values

lime_explainer = LimeTabularExplainer(
    X_train_np,
    feature_names=feature_labels,
    mode="regression"
)

lime_importance = np.zeros(len(feature_labels))
n_samples = min(50, len(X_test_np))

for i in range(n_samples):

    exp = lime_explainer.explain_instance(
        X_test_np[i],
        model.predict,  # ✅ DOĞRU MODEL
        num_features=len(feature_labels)
    )

    for feat, val in exp.as_list():
        clean_feat = feat.split(" ")[0]
        if clean_feat in feature_labels:
            idx = feature_labels.index(clean_feat)
            lime_importance[idx] += abs(val)

lime_importance /= n_samples

lime_rank_df = pd.DataFrame({
    "Feature": feature_labels,
    "LIME_Importance": lime_importance
}).sort_values("LIME_Importance", ascending=False)

lime_rank_df["LIME_Rank"] = range(1, len(lime_rank_df) + 1)

print("\n=========== LIME IMPORTANCE ===========")
print(lime_rank_df.round(4))

# --------------------------------------------------
# 3️⃣ MERGE & RANK DIFFERENCE
# --------------------------------------------------

plot_df = shap_rank_df.merge(lime_rank_df, on="Feature")
plot_df["Rank_Diff"] = abs(plot_df["SHAP_Rank"] - plot_df["LIME_Rank"])
rank_df = plot_df.sort_values("Rank_Diff")

print("\n=========== FEATURE RANKS & DIFFERENCES ===========")
print(rank_df[["Feature", "SHAP_Rank", "LIME_Rank", "Rank_Diff"]].to_string(index=False))

# --------------------------------------------------
# 4️⃣ VISUALIZATION
# --------------------------------------------------

plt.figure(figsize=(10, 6))

colors = plt.cm.viridis(np.linspace(0, 1, len(rank_df)))

bars = plt.barh(
    rank_df["Feature"],
    rank_df["Rank_Diff"],
    color=colors,
    edgecolor="black"
)

plt.title(
    "SHAP–LIME Rank Difference\n(Lower = Higher Consistency)",
    fontsize=14,
    fontweight="bold"
)

plt.xlabel("Absolute Rank Difference", fontsize=12)
plt.ylabel("Feature", fontsize=12)

for bar, shp, lim in zip(bars, rank_df["SHAP_Rank"], rank_df["LIME_Rank"]):
    width = bar.get_width()
    plt.text(
        width + 0.05,
        bar.get_y() + bar.get_height()/2,
        f"Diff:{width:.0f} (S:{shp}, L:{lim})",
        va="center",
        fontsize=9
    )

# --------------------------------------------------
# 5️⃣ SPEARMAN CONSISTENCY
# --------------------------------------------------

rho, pval = spearmanr(plot_df["SHAP_Rank"], plot_df["LIME_Rank"])

plt.text(
    x=max(rank_df["Rank_Diff"]) * 0.55,
    y=0,
    s=f"Spearman ρ: {rho:.3f}\np-value: {pval:.4f}",
    fontsize=11,
    fontweight="bold",
    color="red"
)

plt.tight_layout()
plt.show()

print("\n=========== RANK CONSISTENCY ===========")
print(f"Spearman ρ: {rho:.3f}")
print(f"p-value: {pval:.4f}")

In [ ]:
# ============================================================
# PANEL ML + TEMPORAL DRIFT + REGIME ANALYSIS (Q1-READY)
# ============================================================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import GroupKFold
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.cluster import KMeans
from sklearn.metrics import r2_score
from sklearn.inspection import PartialDependenceDisplay

import warnings
warnings.filterwarnings("ignore")

# ============================================================
# LOAD PANEL DATA
# ============================================================

df = pd.read_csv("GII.csv", sep=";", encoding="latin1")
df.columns = df.columns.str.strip()

target = "Score"
features = [c for c in df.columns if c not in ["Country", "Year", target]]

# ============================================================
# STANDARDIZATION
# ============================================================

scaler_X = StandardScaler()
scaler_y = StandardScaler()

X_std = scaler_X.fit_transform(df[features])
y_std = scaler_y.fit_transform(df[[target]]).ravel()

df_std = df.copy()
df_std[features] = X_std
df_std[target] = y_std

# ============================================================
# TIME-AWARE CV (GroupKFold by Country)
# ============================================================

gkf = GroupKFold(n_splits=5)
model = GradientBoostingRegressor(n_estimators=400, random_state=42)

r2_scores = []

for train_idx, test_idx in gkf.split(
        df_std[features], df_std[target], groups=df_std["Country"]):

    X_train, X_test = df_std.iloc[train_idx][features], df_std.iloc[test_idx][features]
    y_train, y_test = df_std.iloc[train_idx][target], df_std.iloc[test_idx][target]

    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    r2_scores.append(r2_score(y_test, preds))

print("\nPanel Cross-Validated R² Scores:", np.round(r2_scores,3))
print("Mean CV R²:", round(np.mean(r2_scores),3))

In [ ]:
# ============================================================
# HEATMAP VERSION
# ============================================================

plt.figure(figsize=(12,7))

sns.heatmap(
    cluster_means,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5
)

plt.title("Cluster-Level SHAP Contribution Matrix", fontsize=15)
plt.xlabel("Innovation Dimensions")
plt.ylabel("Innovation Regimes")
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# SHAP INTERACTION VALUES (2D HEATMAP)
# ============================================================

import seaborn as sns

# Compute interaction values
explainer = shap.Explainer(model, df_std[features])
shap_interaction = explainer(df_std[features]).values

# SHAP interaction needs TreeExplainer for full interaction matrix
explainer_tree = shap.TreeExplainer(model)
interaction_values = explainer_tree.shap_interaction_values(df_std[features])

# Mean absolute interaction matrix
interaction_matrix = np.abs(interaction_values).mean(axis=0)

interaction_df = pd.DataFrame(
    interaction_matrix,
    index=features,
    columns=features
)

plt.figure(figsize=(10,8))
sns.heatmap(
    interaction_df,
    annot=True,
    fmt=".3f",
    cmap="coolwarm",
    square=True
)

plt.title("SHAP Interaction Matrix (Mean Absolute Effects)")
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.show()

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

quantiles = [0.25, 0.50, 0.75]
quantile_shap = {}

n_boot = 30  # bootstrap runs

all_shap = []

# ------------------------------------------------------------
# BOOTSTRAP EXTRA TREES MODELS
# ------------------------------------------------------------

for i in range(n_boot):
    
    idx = np.random.choice(len(df_std), len(df_std), replace=True)
    X_boot = df_std[features].iloc[idx]
    y_boot = df_std[target].iloc[idx]

    model = ExtraTreesRegressor(
        n_estimators=400,
        random_state=i,
        n_jobs=-1
    )
    
    model.fit(X_boot, y_boot)

    explainer = shap.Explainer(model, X_boot)
    shap_vals = explainer(X_boot)

    mean_importance = np.abs(shap_vals.values).mean(axis=0)
    all_shap.append(mean_importance)

all_shap = np.array(all_shap)

# ------------------------------------------------------------
# COMPUTE QUANTILE IMPORTANCE
# ------------------------------------------------------------

for q in quantiles:
    quantile_shap[q] = np.quantile(all_shap, q, axis=0)

quantile_df = pd.DataFrame(
    quantile_shap,
    index=features
)

# Normalize for comparability
quantile_df = quantile_df.div(quantile_df.sum(axis=0), axis=1)

# ------------------------------------------------------------
# PRINT NUMERICAL OUTPUT
# ------------------------------------------------------------

print("\n==============================")
print("EXTRA TREES – QUANTILE SHAP IMPORTANCE")
print("==============================\n")
print(quantile_df.round(4))

# ------------------------------------------------------------
# PLOT WITH VALUE LABELS
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 9))

bars = quantile_df.plot(
    kind="bar",
    ax=ax,
    edgecolor="black"
)

plt.title(
    "Extra Trees – Quantile-Specific SHAP Importance\n"
    "Distributional Heterogeneity in Innovation Drivers",
    fontsize=16,
    weight="bold"
)

plt.ylabel("Relative Mean |SHAP| Contribution", fontsize=14)
plt.xticks(rotation=90, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.3)

# ------------------------------------------------------------
# ADD 90-DEGREE VALUE LABELS
# ------------------------------------------------------------

for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=10,
            rotation=90
        )

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import entropy
from scipy.spatial.distance import jensenshannon

sns.set_style("whitegrid")

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

n_boot = 40
n_mc = 100
all_shap = []
all_local_shap = []

# ------------------------------------------------------------
# BOOTSTRAP MODEL + SHAP
# ------------------------------------------------------------

for i in range(n_boot):

    idx = np.random.choice(len(df_std), len(df_std), replace=True)
    X_boot = df_std[features].iloc[idx]
    y_boot = df_std[target].iloc[idx]

    model = ExtraTreesRegressor(
        n_estimators=400,
        random_state=i,
        n_jobs=-1
    )

    model.fit(X_boot, y_boot)

    explainer = shap.TreeExplainer(model)
    shap_vals = explainer.shap_values(X_boot)

    all_shap.append(np.abs(shap_vals).mean(axis=0))
    all_local_shap.append(shap_vals)

all_shap = np.array(all_shap)
global_mean = all_shap.mean(axis=0)

# ============================================================
# 1️⃣ SHAP ENTROPY INDEX
# ============================================================

prob_dist = global_mean / global_mean.sum()
shap_entropy = entropy(prob_dist)

print("\n🔥 SHAP ENTROPY INDEX:", round(shap_entropy, 4))

plt.figure(figsize=(6,4))
bars = plt.bar(["SHAP Entropy"], [shap_entropy], color="crimson")

for bar in bars:
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height(),
             f"{shap_entropy:.4f}",
             ha='center', va='bottom', fontsize=12)

plt.title("SHAP Entropy Index", fontsize=13)
plt.tight_layout()
plt.show()

# ============================================================
# 2️⃣ FEATURE INTERACTION SURFACE
# ============================================================

model_final = ExtraTreesRegressor(n_estimators=400, random_state=0)
model_final.fit(df_std[features], df_std[target])

explainer_final = shap.TreeExplainer(model_final)
interaction_values = explainer_final.shap_interaction_values(df_std[features])

interaction_matrix = np.abs(interaction_values).mean(axis=0)

print("\n🔥 INTERACTION MATRIX (Mean Absolute)")
print(pd.DataFrame(interaction_matrix, 
                   index=features, 
                   columns=features).round(4))

plt.figure(figsize=(10,8))

sns.heatmap(
    interaction_matrix,
    xticklabels=features,
    yticklabels=features,
    cmap="magma",
    annot=True,
    fmt=".2f",
    linewidths=0.5
)

plt.title("SHAP Feature Interaction Surface", fontsize=14)
plt.tight_layout()
plt.show()

# ============================================================
# 3️⃣ BAYESIAN UNCERTAINTY BAND
# ============================================================

posterior_mean = all_shap.mean(axis=0)
posterior_std = all_shap.std(axis=0)

print("\n🔥 POSTERIOR MEAN SHAP")
print(pd.Series(posterior_mean, index=features).round(4))

plt.figure(figsize=(10,6))

bars = plt.bar(features, posterior_mean, 
               yerr=1.96*posterior_std,
               capsize=5,
               color="royalblue")

for i, bar in enumerate(bars):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height(),
             f"{posterior_mean[i]:.3f}",
             ha='center', va='bottom', fontsize=9)

plt.xticks(rotation=45)
plt.title("Bayesian Posterior Approximation – SHAP Importance")
plt.tight_layout()
plt.show()

# ============================================================
# 4️⃣ GLOBAL vs LOCAL SHAP DIVERGENCE
# ============================================================

local_stack = np.vstack(all_local_shap)
local_mean_dist = np.abs(local_stack).mean(axis=0)
local_prob = local_mean_dist / local_mean_dist.sum()

js_divergence = jensenshannon(prob_dist, local_prob)

print("\n🔥 Global vs Local SHAP Jensen-Shannon Divergence:",
      round(js_divergence, 4))

# ============================================================
# 5️⃣ CAUSAL SHAP (Permutation Proxy)
# ============================================================

causal_effects = []

for feature in features:

    X_perm = df_std.copy()
    X_perm[feature] = np.random.permutation(X_perm[feature])

    model_perm = ExtraTreesRegressor(n_estimators=300, random_state=0)
    model_perm.fit(X_perm[features], df_std[target])

    explainer_perm = shap.TreeExplainer(model_perm)
    shap_perm = explainer_perm.shap_values(X_perm[features])

    causal_effects.append(np.abs(shap_perm).mean())

print("\n🔥 Causal SHAP Proxy Effects")
print(pd.Series(causal_effects, index=features).round(4))

plt.figure(figsize=(10,6))

bars = plt.bar(features, causal_effects, color="darkorange")

for i, bar in enumerate(bars):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height(),
             f"{causal_effects[i]:.3f}",
             ha='center', va='bottom', fontsize=9)

plt.xticks(rotation=45)
plt.title("Permutation-Based Causal SHAP Proxy")
plt.tight_layout()
plt.show()

# ============================================================
# 6️⃣ MONTE CARLO SENSITIVITY
# ============================================================

mc_results = []

for i in range(n_mc):

    noise = np.random.normal(0, 0.01, df_std[features].shape)
    X_mc = df_std[features] + noise

    model_mc = ExtraTreesRegressor(n_estimators=200, random_state=i)
    model_mc.fit(X_mc, df_std[target])

    explainer_mc = shap.TreeExplainer(model_mc)
    shap_mc = explainer_mc.shap_values(X_mc)

    mc_results.append(np.abs(shap_mc).mean(axis=0))

mc_results = np.array(mc_results)
mc_std = mc_results.std(axis=0)

print("\n🔥 Monte Carlo Sensitivity (Std of SHAP):")
print(pd.Series(mc_std, index=features).round(4))

plt.figure(figsize=(10,6))

bars = plt.bar(features, mc_std, color="seagreen")

for i, bar in enumerate(bars):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height(),
             f"{mc_std[i]:.4f}",
             ha='center', va='bottom', fontsize=9)

plt.xticks(rotation=45)
plt.title("Monte Carlo SHAP Sensitivity")
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesRegressor
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy

sns.set(style="whitegrid", font_scale=1.1)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

model = ExtraTreesRegressor(n_estimators=500, random_state=0)
model.fit(df_std[features], df_std[target])

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(df_std[features])
interaction_vals = explainer.shap_interaction_values(df_std[features])

interaction_matrix = np.abs(interaction_vals).mean(axis=0)

# ============================================================
# 1️⃣ LAPLACIAN GRAPH DYNAMICS
# ============================================================

G = nx.from_numpy_array(interaction_matrix)
L = nx.laplacian_matrix(G).toarray()

eigenvalues, eigenvectors = eigh(L)
sorted_eigs = sorted(eigenvalues)

print("\nLAPLACIAN EIGENVALUES:")
for i, val in enumerate(sorted_eigs):
    print(f"λ{i+1}: {val:.6f}")

plt.figure(figsize=(9,5))
colors = plt.cm.plasma(np.linspace(0,1,len(sorted_eigs)))

for i, val in enumerate(sorted_eigs):
    plt.scatter(i+1, val, color=colors[i], s=80)
    plt.text(i+1, val, f"{val:.3f}", ha='center', va='bottom')

plt.plot(range(1,len(sorted_eigs)+1), sorted_eigs, color="black", alpha=0.6)
plt.title("Laplacian Spectrum of SHAP Interaction Network")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Eigenvalue")
plt.show()

print("\nAlgebraic Connectivity (λ2):", sorted_eigs[1])

# ============================================================
# 2️⃣ PERSISTENT HOMOLOGY
# ============================================================

distance_matrix = squareform(pdist(shap_vals.T, metric="euclidean"))

print("\nFEATURE DISTANCE MATRIX:")
print(pd.DataFrame(distance_matrix, index=features, columns=features))

plt.figure(figsize=(10,7))
sns.heatmap(
    distance_matrix,
    xticklabels=features,
    yticklabels=features,
    annot=True,
    fmt=".2f",
    cmap="turbo"
)
plt.title("Feature Distance Matrix (Topological Structure)")
plt.show()

threshold = np.median(distance_matrix)
adjacency = (distance_matrix < threshold).astype(int)
G_topo = nx.from_numpy_array(adjacency)
components = nx.number_connected_components(G_topo)

print("\nPersistent Homology Approximation (Betti-0):", components)

# ============================================================
# 3️⃣ SHAP CURVATURE TENSOR
# ============================================================

cov_matrix = np.cov(shap_vals.T)
eigvals_cov, eigvecs_cov = eigh(cov_matrix)

curvature_proxy = eigvals_cov / eigvals_cov.sum()
sorted_curv = sorted(curvature_proxy, reverse=True)

print("\nCURVATURE EIGENVALUES:")
for i, val in enumerate(sorted_curv):
    print(f"Component {i+1}: {val:.6f}")

plt.figure(figsize=(9,5))
colors = plt.cm.viridis(np.linspace(0,1,len(sorted_curv)))

for i, val in enumerate(sorted_curv):
    plt.bar(i+1, val, color=colors[i])
    plt.text(i+1, val, f"{val:.3f}", ha='center', va='bottom')

plt.title("SHAP Curvature Spectrum")
plt.xlabel("Component")
plt.ylabel("Normalized Eigenvalue")
plt.show()

print("\nCurvature Concentration Ratio:",
      max(curvature_proxy))

# ============================================================
# 4️⃣ INFORMATION ENTROPY FLOW
# ============================================================

prob_matrix = interaction_matrix / interaction_matrix.sum()
node_entropy = entropy(prob_matrix, axis=1)

print("\nNODE ENTROPY VALUES:")
for f, val in zip(features, node_entropy):
    print(f"{f}: {val:.6f}")

plt.figure(figsize=(11,6))
colors = plt.cm.rainbow(np.linspace(0,1,len(features)))

bars = plt.bar(features, node_entropy, color=colors)

for bar, val in zip(bars, node_entropy):
    plt.text(bar.get_x() + bar.get_width()/2,
             bar.get_height(),
             f"{val:.3f}",
             ha='center',
             va='bottom')

plt.xticks(rotation=45)
plt.title("Node-wise Information Entropy")
plt.show()

entropy_flow_rate = np.gradient(node_entropy)

print("\nENTROPY FLOW DIFFERENTIAL:")
for f, val in zip(features, entropy_flow_rate):
    print(f"{f}: {val:.6f}")

plt.figure(figsize=(9,5))
colors = plt.cm.cool(np.linspace(0,1,len(entropy_flow_rate)))

for i, val in enumerate(entropy_flow_rate):
    plt.scatter(i+1, val, color=colors[i], s=80)
    plt.text(i+1, val, f"{val:.3f}", ha='center', va='bottom')

plt.plot(range(1,len(entropy_flow_rate)+1),
         entropy_flow_rate,
         color="black",
         alpha=0.6)

plt.title("Entropy Flow Differential")
plt.xlabel("Feature Index")
plt.ylabel("Gradient Value")
plt.show()

# ============================================================
# 5️⃣ QUANTUM SHAP
# ============================================================

density_matrix = cov_matrix / np.trace(cov_matrix)
eigenvals_density, _ = eigh(density_matrix)

sorted_density = sorted(eigenvals_density, reverse=True)

von_neumann_entropy = -np.sum(
    eigenvals_density * np.log(eigenvals_density + 1e-12)
)

print("\nDensity Matrix Eigenvalues:")
for i, val in enumerate(sorted_density):
    print(f"ρ{i+1}: {val:.6f}")

print("\nQuantum SHAP Von Neumann Entropy:",
      round(von_neumann_entropy, 6))

plt.figure(figsize=(9,5))
colors = plt.cm.inferno(np.linspace(0,1,len(sorted_density)))

for i, val in enumerate(sorted_density):
    plt.bar(i+1, val, color=colors[i])
    plt.text(i+1, val, f"{val:.3f}", ha='center', va='bottom')

plt.title("Density Matrix Eigen-Spectrum (Quantum SHAP)")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Density Eigenvalue")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesRegressor
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy
from matplotlib.patches import Patch

sns.set(style="whitegrid", font_scale=1.1)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

model = ExtraTreesRegressor(n_estimators=500, random_state=0)
model.fit(df_std[features], df_std[target])

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(df_std[features])
interaction_vals = explainer.shap_interaction_values(df_std[features])

interaction_matrix = np.abs(interaction_vals).mean(axis=0)

# ============================================================
# 🎨 FEATURE → COLOR MAP (GLOBAL – TÜM FIGURE’LAR İÇİN)
# ============================================================

palette = sns.color_palette("tab10", len(features))
feature_colors = dict(zip(features, palette))

legend_elements = [
    Patch(facecolor=feature_colors[f], label=f)
    for f in features
]

# ============================================================
# 1️⃣ LAPLACIAN GRAPH DYNAMICS
# ============================================================

G = nx.from_numpy_array(interaction_matrix)
L = nx.laplacian_matrix(G).toarray()

eigenvalues, _ = eigh(L)
sorted_eigs = sorted(eigenvalues)

plt.figure(figsize=(9,5))

for i, val in enumerate(sorted_eigs):
    plt.scatter(
        i + 1,
        val,
        color=feature_colors[features[i % len(features)]],
        s=80
    )
    plt.text(i + 1, val, f"{val:.3f}", ha='center', va='bottom')

plt.plot(range(1, len(sorted_eigs) + 1), sorted_eigs, color="black", alpha=0.6)
plt.title("Laplacian Spectrum of SHAP Interaction Network")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Eigenvalue")

plt.legend(
    handles=legend_elements,
    title="Features",
    fontsize=9,
    title_fontsize=10
)

plt.show()

# ============================================================
# 2️⃣ PERSISTENT HOMOLOGY (DISTANCE HEATMAP)
# ============================================================

distance_matrix = squareform(pdist(shap_vals.T, metric="euclidean"))

plt.figure(figsize=(10,7))
sns.heatmap(
    distance_matrix,
    xticklabels=features,
    yticklabels=features,
    cmap="turbo",
    annot=True,
    fmt=".2f"
)
plt.title("Feature Distance Matrix (Topological Structure)")
plt.show()

# ============================================================
# 3️⃣ SHAP CURVATURE TENSOR
# ============================================================

cov_matrix = np.cov(shap_vals.T)
eigvals_cov, _ = eigh(cov_matrix)

curvature_proxy = eigvals_cov / eigvals_cov.sum()
sorted_curv = sorted(curvature_proxy, reverse=True)

plt.figure(figsize=(9,5))

for i, val in enumerate(sorted_curv):
    plt.bar(
        i + 1,
        val,
        color=feature_colors[features[i % len(features)]]
    )
    plt.text(i + 1, val, f"{val:.3f}", ha='center', va='bottom')

plt.title("SHAP Curvature Spectrum")
plt.xlabel("Component")
plt.ylabel("Normalized Eigenvalue")

plt.legend(
    handles=legend_elements,
    title="Features",
    fontsize=9,
    title_fontsize=10
)

plt.show()

# ============================================================
# 4️⃣ INFORMATION ENTROPY FLOW
# ============================================================

prob_matrix = interaction_matrix / interaction_matrix.sum()
node_entropy = entropy(prob_matrix, axis=1)

plt.figure(figsize=(11,6))

bars = plt.bar(
    features,
    node_entropy,
    color=[feature_colors[f] for f in features]
)

for bar, val in zip(bars, node_entropy):
    plt.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height(),
        f"{val:.3f}",
        ha='center',
        va='bottom'
    )

plt.xticks(rotation=45)
plt.title("Node-wise Information Entropy")

plt.legend(
    handles=legend_elements,
    title="Features",
    fontsize=9,
    title_fontsize=10
)

plt.show()

entropy_flow_rate = np.gradient(node_entropy)

plt.figure(figsize=(9,5))

for i, val in enumerate(entropy_flow_rate):
    plt.scatter(
        i + 1,
        val,
        color=feature_colors[features[i]],
        s=80
    )
    plt.text(i + 1, val, f"{val:.3f}", ha='center', va='bottom')

plt.plot(range(1, len(entropy_flow_rate) + 1),
         entropy_flow_rate,
         color="black",
         alpha=0.6)

plt.title("Entropy Flow Differential")
plt.xlabel("Feature Index")
plt.ylabel("Gradient Value")

plt.legend(
    handles=legend_elements,
    title="Features",
    fontsize=9,
    title_fontsize=10
)

plt.show()

# ============================================================
# 5️⃣ QUANTUM SHAP
# ============================================================

density_matrix = cov_matrix / np.trace(cov_matrix)
eigenvals_density, _ = eigh(density_matrix)

sorted_density = sorted(eigenvals_density, reverse=True)

von_neumann_entropy = -np.sum(
    eigenvals_density * np.log(eigenvals_density + 1e-12)
)

plt.figure(figsize=(9,5))

for i, val in enumerate(sorted_density):
    plt.bar(
        i + 1,
        val,
        color=feature_colors[features[i % len(features)]]
    )
    plt.text(i + 1, val, f"{val:.3f}", ha='center', va='bottom')

plt.title("Density Matrix Eigen-Spectrum (Quantum SHAP)")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Density Eigenvalue")

plt.legend(
    handles=legend_elements,
    title="Features",
    fontsize=9,
    title_fontsize=10
)

plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesRegressor
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy
from matplotlib.patches import Patch

sns.set(style="whitegrid", font_scale=1.1)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

model = ExtraTreesRegressor(n_estimators=500, random_state=0)
model.fit(df_std[features], df_std[target])

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(df_std[features])
interaction_vals = explainer.shap_interaction_values(df_std[features])

interaction_matrix = np.abs(interaction_vals).mean(axis=0)

# ============================================================
# 🎨 FEATURE → COLOR MAP (GLOBAL)
# ============================================================

palette = sns.color_palette("tab10", len(features))
feature_colors = dict(zip(features, palette))

legend_elements = [
    Patch(facecolor=feature_colors[f], label=f)
    for f in features
]

# ============================================================
# 1️⃣ LAPLACIAN GRAPH DYNAMICS
# ============================================================

G = nx.from_numpy_array(interaction_matrix)
L = nx.laplacian_matrix(G).toarray()

eigenvalues, _ = eigh(L)
sorted_eigs = sorted(eigenvalues)

plt.figure(figsize=(9,5))

for i, val in enumerate(sorted_eigs):
    plt.scatter(
        i + 1, val,
        color=feature_colors[features[i % len(features)]],
        s=80
    )
    plt.text(i + 1, val, f"{val:.4f}", ha='center', va='bottom', fontsize=9)

plt.plot(range(1, len(sorted_eigs)+1), sorted_eigs, color="black", alpha=0.6)
plt.title("Laplacian Spectrum of SHAP Interaction Network")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Eigenvalue")
plt.legend(handles=legend_elements, title="Features", fontsize=9)
plt.show()

# ============================================================
# 2️⃣ PERSISTENT HOMOLOGY (DISTANCE HEATMAP)
# ============================================================

distance_matrix = squareform(pdist(shap_vals.T, metric="euclidean"))

plt.figure(figsize=(10,7))
sns.heatmap(
    distance_matrix,
    xticklabels=features,
    yticklabels=features,
    annot=True,
    fmt=".2f",
    cmap="turbo"
)
plt.title("Feature Distance Matrix (Topological Structure)")
plt.show()

# ============================================================
# 3️⃣ SHAP CURVATURE TENSOR
# ============================================================

cov_matrix = np.cov(shap_vals.T)
eigvals_cov, _ = eigh(cov_matrix)

curvature_proxy = eigvals_cov / eigvals_cov.sum()
sorted_curv = sorted(curvature_proxy, reverse=True)

plt.figure(figsize=(9,5))

for i, val in enumerate(sorted_curv):
    plt.bar(i + 1, val,
            color=feature_colors[features[i % len(features)]])
    plt.text(i + 1, val, f"{val:.4f}",
             ha='center', va='bottom', fontsize=9)

plt.title("SHAP Curvature Spectrum")
plt.xlabel("Component")
plt.ylabel("Normalized Eigenvalue")
plt.legend(handles=legend_elements, title="Features", fontsize=9)
plt.show()

# ============================================================
# 4️⃣ INFORMATION ENTROPY FLOW
# ============================================================

prob_matrix = interaction_matrix / interaction_matrix.sum()
node_entropy = entropy(prob_matrix, axis=1)

plt.figure(figsize=(11,6))

bars = plt.bar(
    features,
    node_entropy,
    color=[feature_colors[f] for f in features]
)

for bar, val in zip(bars, node_entropy):
    plt.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{val:.4f}",
        ha='center', va='bottom', fontsize=9
    )

plt.xticks(rotation=45)
plt.title("Node-wise Information Entropy")
plt.legend(handles=legend_elements, title="Features", fontsize=9)
plt.show()

# ============================================================
# ENTROPY FLOW DIFFERENTIAL
# ============================================================

entropy_flow_rate = np.gradient(node_entropy)

plt.figure(figsize=(9,5))

for i, val in enumerate(entropy_flow_rate):
    plt.scatter(
        i + 1, val,
        color=feature_colors[features[i]],
        s=80
    )
    plt.text(i + 1, val, f"{val:.4f}",
             ha='center', va='bottom', fontsize=9)

plt.plot(range(1, len(entropy_flow_rate)+1),
         entropy_flow_rate, color="black", alpha=0.6)

plt.title("Entropy Flow Differential")
plt.xlabel("Feature Index")
plt.ylabel("Gradient Value")
plt.legend(handles=legend_elements, title="Features", fontsize=9)
plt.show()

# ============================================================
# 5️⃣ QUANTUM SHAP
# ============================================================

density_matrix = cov_matrix / np.trace(cov_matrix)
eigenvals_density, _ = eigh(density_matrix)

sorted_density = sorted(eigenvals_density, reverse=True)

von_neumann_entropy = -np.sum(
    eigenvals_density * np.log(eigenvals_density + 1e-12)
)

plt.figure(figsize=(9,5))

for i, val in enumerate(sorted_density):
    plt.bar(i + 1, val,
            color=feature_colors[features[i % len(features)]])
    plt.text(i + 1, val, f"{val:.4f}",
             ha='center', va='bottom', fontsize=9)

plt.title("Density Matrix Eigen-Spectrum (Quantum SHAP)")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Density Eigenvalue")
plt.legend(handles=legend_elements, title="Features", fontsize=9)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesRegressor
from scipy.linalg import eigh
from scipy.spatial.distance import pdist, squareform
from scipy.stats import entropy
from matplotlib.patches import Patch

sns.set(style="whitegrid", font_scale=1.1)

# ------------------------------------------------------------
# MODEL EĞİTİMİ
# ------------------------------------------------------------

model = ExtraTreesRegressor(n_estimators=500, random_state=0)
model.fit(df_std[features], df_std[target])

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(df_std[features])
interaction_vals = explainer.shap_interaction_values(df_std[features])

interaction_matrix = np.abs(interaction_vals).mean(axis=0)

# ------------------------------------------------------------
# RENK HARİTASI
# ------------------------------------------------------------

palette = sns.color_palette("tab10", len(features))
feature_colors = dict(zip(features, palette))

legend_elements = [
    Patch(facecolor=feature_colors[f], label=f)
    for f in features
]

# ============================================================
# 1️⃣ LAPLACIAN SPECTRUM
# ============================================================

G = nx.from_numpy_array(interaction_matrix)
L = nx.laplacian_matrix(G).toarray()

eigenvalues, _ = eigh(L)
sorted_eigs = np.sort(eigenvalues)

print("\n🔹 Laplacian Eigenvalues:")
for i, val in enumerate(sorted_eigs, 1):
    print(f"Eigenvalue {i}: {val:.6f}")

plt.figure(figsize=(9,5))
for i, val in enumerate(sorted_eigs):
    plt.scatter(i+1, val, s=80, color=feature_colors[features[i % len(features)]])
    plt.text(i+1, val, f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.plot(range(1, len(sorted_eigs)+1), sorted_eigs, color="black")
plt.title("Laplacian Spectrum of SHAP Interaction Network")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Eigenvalue")
plt.legend(handles=legend_elements)
plt.show()

# ============================================================
# 2️⃣ TOPOLOJİK MESAFE MATRİSİ
# ============================================================

distance_matrix = squareform(pdist(shap_vals.T, metric="euclidean"))

print("\n🔹 Feature Distance Matrix:")
print(pd.DataFrame(distance_matrix, index=features, columns=features).round(4))

plt.figure(figsize=(10,7))
sns.heatmap(distance_matrix,
            xticklabels=features,
            yticklabels=features,
            annot=True,
            fmt=".2f",
            cmap="turbo")
plt.title("Feature Distance Matrix (Topological Structure)")
plt.show()

# ============================================================
# 3️⃣ SHAP CURVATURE SPECTRUM
# ============================================================

cov_matrix = np.cov(shap_vals.T)
eigvals_cov, _ = eigh(cov_matrix)

curvature_proxy = eigvals_cov / eigvals_cov.sum()
sorted_curv = np.sort(curvature_proxy)[::-1]

print("\n🔹 SHAP Curvature Eigenvalues:")
for i, val in enumerate(sorted_curv, 1):
    print(f"Component {i}: {val:.6f}")

plt.figure(figsize=(9,5))
for i, val in enumerate(sorted_curv):
    plt.bar(i+1, val, color=feature_colors[features[i % len(features)]])
    plt.text(i+1, val, f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.title("SHAP Curvature Spectrum")
plt.xlabel("Component")
plt.ylabel("Normalized Eigenvalue")
plt.legend(handles=legend_elements)
plt.show()

# ============================================================
# 4️⃣ NODE-WISE INFORMATION ENTROPY
# ============================================================

prob_matrix = interaction_matrix / interaction_matrix.sum()
node_entropy = entropy(prob_matrix, axis=1)

print("\n🔹 Node-wise Entropy Values:")
for f, val in zip(features, node_entropy):
    print(f"{f}: {val:.6f}")

plt.figure(figsize=(11,6))
bars = plt.bar(features, node_entropy,
               color=[feature_colors[f] for f in features])

for bar, val in zip(bars, node_entropy):
    plt.text(bar.get_x()+bar.get_width()/2,
             bar.get_height(),
             f"{val:.4f}",
             ha="center", va="bottom", fontsize=9)

plt.xticks(rotation=45)
plt.title("Node-wise Information Entropy")
plt.legend(handles=legend_elements)
plt.show()

# ============================================================
# ENTROPY FLOW DIFFERENTIAL
# ============================================================

entropy_flow_rate = np.gradient(node_entropy)

print("\n🔹 Entropy Flow Gradient:")
for f, val in zip(features, entropy_flow_rate):
    print(f"{f}: {val:.6f}")

plt.figure(figsize=(9,5))
for i, val in enumerate(entropy_flow_rate):
    plt.scatter(i+1, val, s=80, color=feature_colors[features[i]])
    plt.text(i+1, val, f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.plot(range(1, len(entropy_flow_rate)+1),
         entropy_flow_rate, color="black")
plt.title("Entropy Flow Differential")
plt.xlabel("Feature Index")
plt.ylabel("Gradient Value")
plt.legend(handles=legend_elements)
plt.show()

# ============================================================
# 5️⃣ QUANTUM SHAP – DENSITY MATRIX
# ============================================================

density_matrix = cov_matrix / np.trace(cov_matrix)
eigenvals_density, _ = eigh(density_matrix)
sorted_density = np.sort(eigenvals_density)[::-1]

von_neumann_entropy = -np.sum(
    eigenvals_density * np.log(eigenvals_density + 1e-12)
)

print("\n🔹 Density Matrix Eigenvalues:")
for i, val in enumerate(sorted_density, 1):
    print(f"Eigenvalue {i}: {val:.6f}")

print(f"\n🔹 Von Neumann Entropy: {von_neumann_entropy:.6f}")

plt.figure(figsize=(9,5))
for i, val in enumerate(sorted_density):
    plt.bar(i+1, val, color=feature_colors[features[i % len(features)]])
    plt.text(i+1, val, f"{val:.4f}", ha="center", va="bottom", fontsize=9)

plt.title("Density Matrix Eigen-Spectrum (Quantum SHAP)")
plt.xlabel("Eigenvalue Index")
plt.ylabel("Density Eigenvalue")
plt.legend(handles=legend_elements)
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import ExtraTreesRegressor
from scipy.linalg import eigh, expm
from scipy.spatial.distance import pdist, squareform
from numpy.linalg import eig
from gudhi import RipsComplex
from gudhi import plot_persistence_barcode

sns.set(style="whitegrid", font_scale=1.1)

# ------------------------------------------------------------
# TRAIN MODEL
# ------------------------------------------------------------

model = ExtraTreesRegressor(n_estimators=500, random_state=0)
model.fit(df_std[features], df_std[target])

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(df_std[features])
interaction_vals = explainer.shap_interaction_values(df_std[features])

interaction_matrix = np.abs(interaction_vals).mean(axis=0)

# ============================================================
# 1️⃣ RICCI CURVATURE
# ============================================================

G = nx.from_numpy_array(interaction_matrix)
G = nx.relabel_nodes(G, dict(enumerate(features)))

ricci_curvature = {}

for node in G.nodes():
    neighbors = list(G.neighbors(node))
    deg = G.degree(node)
    avg_neighbor_deg = np.mean([G.degree(n) for n in neighbors])
    ricci_curvature[node] = 1 - (avg_neighbor_deg / deg)

ricci_series = pd.Series(ricci_curvature).sort_values(ascending=False)

print("\nRICCI CURVATURE VALUES:")
print(ricci_series)

plt.figure(figsize=(10,6))
colors = plt.cm.rainbow(np.linspace(0,1,len(ricci_series)))
bars = plt.bar(ricci_series.index, ricci_series.values, color=colors)

for bar, val in zip(bars, ricci_series.values):
    plt.text(bar.get_x()+bar.get_width()/2,
             bar.get_height(),
             f"{val:.3f}",
             ha='center', va='bottom')

plt.xticks(rotation=45)
plt.title("Ricci Curvature Approximation on SHAP Network")
plt.show()

# ============================================================
# 2️⃣ PERSISTENT HOMOLOGY
# ============================================================

distance_matrix = squareform(pdist(shap_vals.T))

print("\nDISTANCE MATRIX:")
print(pd.DataFrame(distance_matrix, index=features, columns=features))

plt.figure(figsize=(10,7))
sns.heatmap(distance_matrix,
            xticklabels=features,
            yticklabels=features,
            annot=True,
            fmt=".2f",
            cmap="turbo")
plt.title("Feature Distance Matrix (Topological Structure)")
plt.show()

rips = RipsComplex(distance_matrix=distance_matrix,
                   max_edge_length=np.max(distance_matrix))
simplex_tree = rips.create_simplex_tree(max_dimension=2)
diag = simplex_tree.persistence()

print("\nPERSISTENCE DIAGRAM:")
for d in diag:
    print(d)

plot_persistence_barcode(diag)
plt.title("Persistent Homology Barcode (SHAP Space)")
plt.show()

# ============================================================
# 3️⃣ GRAPH HEAT EQUATION
# ============================================================

L = nx.laplacian_matrix(G).toarray()
initial_signal = np.abs(shap_vals).mean(axis=0)

t = 0.5
heat_diffusion = expm(-t * L) @ initial_signal

print("\nHEAT DIFFUSION VALUES:")
for f, val in zip(features, heat_diffusion):
    print(f"{f}: {val:.6f}")

plt.figure(figsize=(10,6))
colors = plt.cm.plasma(np.linspace(0,1,len(features)))
bars = plt.bar(features, heat_diffusion, color=colors)

for bar, val in zip(bars, heat_diffusion):
    plt.text(bar.get_x()+bar.get_width()/2,
             bar.get_height(),
             f"{val:.3f}",
             ha='center', va='bottom')

plt.xticks(rotation=90)
plt.title("Graph Heat Diffusion on SHAP Network")
plt.show()

# ============================================================
# 4️⃣ KOOPMAN OPERATOR
# ============================================================

X_t = shap_vals[:-1]
X_t1 = shap_vals[1:]
K = np.linalg.pinv(X_t) @ X_t1
eigvals_koopman, _ = eig(K)

print("\nKOOPMAN EIGENVALUES:")
for i, val in enumerate(eigvals_koopman):
    print(f"λ{i+1}: {val.real:.6f} + {val.imag:.6f}i")

plt.figure(figsize=(7,7))
colors = plt.cm.cool(np.linspace(0,1,len(eigvals_koopman)))

for i, val in enumerate(eigvals_koopman):
    plt.scatter(val.real, val.imag, color=colors[i], s=80)
    plt.text(val.real, val.imag,
             f"{val.real:.2f}",
             fontsize=8)

plt.axhline(0, color='black')
plt.axvline(0, color='black')
plt.title("Koopman Operator Spectrum")
plt.xlabel("Real")
plt.ylabel("Imaginary")
plt.show()

# ============================================================
# 5️⃣ SHAP FIELD PDE
# ============================================================

dt = 0.1
steps = 20
field = initial_signal.copy()

for _ in range(steps):
    field = field - dt * (L @ field)

print("\nFINAL SHAP FIELD VALUES:")
for f, val in zip(features, field):
    print(f"{f}: {val:.6f}")

plt.figure(figsize=(10,6))
colors = plt.cm.viridis(np.linspace(0,1,len(features)))
bars = plt.bar(features, field, color=colors)

for bar, val in zip(bars, field):
    plt.text(bar.get_x()+bar.get_width()/2,
             bar.get_height(),
             f"{val:.3f}",
             ha='center', va='bottom')

plt.xticks(rotation=90)
plt.title("SHAP Field PDE Evolution (Discrete Diffusion)")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import shap
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
from sklearn.ensemble import ExtraTreesRegressor
from scipy.linalg import eigh
from scipy.stats import entropy

# ------------------------------------------------------------
# MODEL TRAIN
# ------------------------------------------------------------

model = ExtraTreesRegressor(n_estimators=500, random_state=0)
model.fit(df_std[features], df_std[target])

explainer = shap.TreeExplainer(model)
shap_vals = explainer.shap_values(df_std[features])
interaction_vals = explainer.shap_interaction_values(df_std[features])

interaction_matrix = np.abs(interaction_vals).mean(axis=0)
mean_importance = np.abs(shap_vals).mean(axis=0)

colors = px.colors.qualitative.Bold

# ============================================================
# HESAPLAMALAR
# ============================================================

potential = mean_importance
gradient = np.gradient(potential)
action = 0.5 * gradient**2 + potential

kinetic = 0.5 * gradient**2
lagrangian = kinetic - potential

paths = 500
path_integral = []
for i in range(len(features)):
    samples = np.random.normal(mean_importance[i],
                               0.1*mean_importance[i] if mean_importance[i]!=0 else 0.001,
                               paths)
    path_integral.append(np.mean(np.exp(-samples)))

L = np.diag(interaction_matrix.sum(axis=1)) - interaction_matrix
ricci_flow = interaction_matrix - 0.1 * L
ricci_strength = ricci_flow.mean(axis=1)

gauge_field = mean_importance / np.linalg.norm(mean_importance)

# ============================================================
# SAYISAL ÇIKTILAR
# ============================================================

print("\n=== HAMILTON–JACOBI ACTION FIELD ===")
for f, v in zip(features, action):
    print(f"{f:<20} : {v:.10f}")

print("\n=== LAGRANGIAN FUNCTIONAL ===")
for f, v in zip(features, lagrangian):
    print(f"{f:<20} : {v:.10f}")

print("\n=== PATH INTEGRAL ===")
for f, v in zip(features, path_integral):
    print(f"{f:<20} : {v:.10f}")

print("\n=== RICCI FLOW STRENGTH ===")
for f, v in zip(features, ricci_strength):
    print(f"{f:<20} : {v:.10f}")

print("\n=== GAUGE FIELD ===")
for f, v in zip(features, gauge_field):
    print(f"{f:<20} : {v:.10f}")

# ============================================================
# TEK SÜTUN DASHBOARD
# ============================================================

fig = make_subplots(
    rows=5, cols=1,
    vertical_spacing=0.05,
    subplot_titles=[
        "Hamilton–Jacobi Action Field",
        "Lagrangian Functional",
        "Path Integral",
        "Ricci Flow Evolution",
        "Gauge-Invariant Field"
    ]
)

# 1️⃣ Action
for i, f in enumerate(features):
    fig.add_trace(
        go.Bar(
            x=[f],
            y=[action[i]],
            marker_color=colors[i % len(colors)],
            text=[f"{action[i]:.6f}"],
            textposition="outside"
        ),
        row=1, col=1
    )

# 2️⃣ Lagrangian
for i, f in enumerate(features):
    fig.add_trace(
        go.Bar(
            x=[f],
            y=[lagrangian[i]],
            marker_color=colors[i % len(colors)],
            text=[f"{lagrangian[i]:.6f}"],
            textposition="outside"
        ),
        row=2, col=1
    )

# 3️⃣ Path Integral
for i, f in enumerate(features):
    fig.add_trace(
        go.Bar(
            x=[f],
            y=[path_integral[i]],
            marker_color=colors[i % len(colors)],
            text=[f"{path_integral[i]:.6f}"],
            textposition="outside"
        ),
        row=3, col=1
    )

# 4️⃣ Ricci Flow
for i, f in enumerate(features):
    fig.add_trace(
        go.Bar(
            x=[f],
            y=[ricci_strength[i]],
            marker_color=colors[i % len(colors)],
            text=[f"{ricci_strength[i]:.6f}"],
            textposition="outside"
        ),
        row=4, col=1
    )

# 5️⃣ Gauge Field
for i, f in enumerate(features):
    fig.add_trace(
        go.Bar(
            x=[f],
            y=[gauge_field[i]],
            marker_color=colors[i % len(colors)],
            text=[f"{gauge_field[i]:.6f}"],
            textposition="outside"
        ),
        row=5, col=1
    )

fig.update_layout(
    height=3800,   # 🔥 daha da büyütüldü
    width=1300,
    showlegend=False,
    title_text="Unified Explainability Physics Dashboard",
    title_x=0.5,
    font=dict(size=18)
)

fig.show()

In [ ]:
from sklearn.ensemble import ExtraTreesRegressor
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import shap

quantiles = [0.25, 0.50, 0.75]
quantile_shap = {}

n_boot = 30  # bootstrap runs

all_shap = []

# ------------------------------------------------------------
# BOOTSTRAP EXTRA TREES MODELS
# ------------------------------------------------------------

for i in range(n_boot):
    
    idx = np.random.choice(len(df_std), len(df_std), replace=True)
    X_boot = df_std[features].iloc[idx]
    y_boot = df_std[target].iloc[idx]

    model = ExtraTreesRegressor(
        n_estimators=400,
        random_state=i,
        n_jobs=-1
    )
    
    model.fit(X_boot, y_boot)

    explainer = shap.Explainer(model, X_boot)
    shap_vals = explainer(X_boot)

    mean_importance = np.abs(shap_vals.values).mean(axis=0)
    all_shap.append(mean_importance)

all_shap = np.array(all_shap)

# ------------------------------------------------------------
# COMPUTE QUANTILE IMPORTANCE
# ------------------------------------------------------------

for q in quantiles:
    quantile_shap[q] = np.quantile(all_shap, q, axis=0)

quantile_df = pd.DataFrame(
    quantile_shap,
    index=features
)

# Normalize for comparability
quantile_df = quantile_df.div(quantile_df.sum(axis=0), axis=1)

# ------------------------------------------------------------
# PRINT NUMERICAL OUTPUT
# ------------------------------------------------------------

print("\n==============================")
print("EXTRA TREES – QUANTILE SHAP IMPORTANCE")
print("==============================\n")
print(quantile_df.round(4))

# ------------------------------------------------------------
# PLOT WITH VALUE LABELS
# ------------------------------------------------------------

fig, ax = plt.subplots(figsize=(14, 7))

bars = quantile_df.plot(
    kind="bar",
    ax=ax,
    edgecolor="black"
)

plt.title(
    "Extra Trees – Quantile-Specific SHAP Importance\n"
    "Distributional Heterogeneity in Innovation Drivers",
    fontsize=16,
    weight="bold"
)

plt.ylabel("Relative Mean |SHAP| Contribution", fontsize=14)
plt.xticks(rotation=45, fontsize=12)
plt.yticks(fontsize=12)
plt.grid(axis="y", linestyle="--", alpha=0.3)

# ------------------------------------------------------------
# ADD 90-DEGREE VALUE LABELS
# ------------------------------------------------------------

for container in ax.containers:
    for bar in container:
        height = bar.get_height()
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            height,
            f"{height:.3f}",
            ha="center",
            va="bottom",
            fontsize=9,
            rotation=90
        )

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# 🔥 FULL WORKING SOBOL GLOBAL VARIANCE DECOMPOSITION
# ============================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.model_selection import train_test_split

# ------------------------------------------------------------
# 1️⃣ MODEL TRAIN
# ------------------------------------------------------------

X = df_std[features]
y = df_std[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

high_model = ExtraTreesRegressor(
    n_estimators=500,
    random_state=42,
    n_jobs=-1
)

high_model.fit(X_train, y_train)

print("✅ Model trained")
print("R² (test):", round(high_model.score(X_test, y_test), 4))

# ------------------------------------------------------------
# 2️⃣ SOBOL SETTINGS
# ------------------------------------------------------------

N = 5000
X_vals = X.values

mins = X_vals.min(axis=0)
maxs = X_vals.max(axis=0)
d = X_vals.shape[1]

A = np.random.uniform(mins, maxs, size=(N, d))
B = np.random.uniform(mins, maxs, size=(N, d))

fA = high_model.predict(A)
fB = high_model.predict(B)

VarY = np.var(fA)

sobol_first = {}
sobol_total = {}

# ------------------------------------------------------------
# 3️⃣ FIRST & TOTAL ORDER
# ------------------------------------------------------------

for i in range(d):

    ABi = A.copy()
    ABi[:, i] = B[:, i]

    fABi = high_model.predict(ABi)

    Si = np.mean(fB * (fABi - fA)) / VarY
    STi = np.mean((fA - fABi) ** 2) / (2 * VarY)

    sobol_first[features[i]] = Si
    sobol_total[features[i]] = STi

sobol_df = pd.DataFrame({
    "First Order": sobol_first,
    "Total Order": sobol_total
}).T

# ------------------------------------------------------------
# 4️⃣ SECOND ORDER (Institutions × Human capital)
# ------------------------------------------------------------

if "Institutions" in features and "Human capital and research" in features:

    i1 = features.index("Institutions")
    i2 = features.index("Human capital and research")

    AB12 = A.copy()
    AB12[:, i1] = B[:, i1]
    AB12[:, i2] = B[:, i2]

    fAB12 = high_model.predict(AB12)

    S1 = sobol_first["Institutions"]
    S2 = sobol_first["Human capital and research"]

    S12 = (np.mean(fA * (fAB12 - fA)) / VarY) - S1 - S2

else:
    S12 = None

# ------------------------------------------------------------
# 5️⃣ PRINT RESULTS
# ------------------------------------------------------------

print("\n🔥 SOBOL RESULTS")
print(sobol_df.round(4))

if S12 is not None:
    print("\n🔥 SECOND-ORDER INTERACTION (Institutions × Human Capital):", round(S12,4))

# ------------------------------------------------------------
# 6️⃣ VISUALIZATION (FIRST & TOTAL)
# ------------------------------------------------------------

plt.figure(figsize=(12,8))

x = np.arange(len(features))
width = 0.35

first_vals = list(sobol_first.values())
total_vals = list(sobol_total.values())

bars1 = plt.bar(x - width/2, first_vals, width, label="First Order")
bars2 = plt.bar(x + width/2, total_vals, width, label="Total Order")

for bar in bars1:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height,
             f"{height:.2f}",
             ha='center', va='bottom', fontsize=8)

for bar in bars2:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2,
             height,
             f"{height:.2f}",
             ha='center', va='bottom', fontsize=8)

plt.xticks(x, features, rotation=90)
plt.ylabel("Sobol Index")
plt.title("Global Sobol Variance Decomposition")
plt.legend()
plt.tight_layout()
plt.show()

# ------------------------------------------------------------
# 7️⃣ SECOND ORDER VISUAL
# ------------------------------------------------------------

if S12 is not None:
    plt.figure(figsize=(5,4))
    plt.bar(["Institutions × Human Capital"], [S12])
    plt.text(0, S12, f"{S12:.3f}", ha='center', va='bottom')
    plt.ylabel("Second-Order Sobol Index")
    plt.title("Nonlinear Complementarity Effect")
    plt.tight_layout()
    plt.show()